# Library

In [2]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import re
import time # For polite scraping delays
import requests
import numpy as np
import json

In [2]:
# # this is for dynamic javascript website
# pip install playwright
# playwright install

# Try to webscrap for Board Members' speeches

In [53]:
# --- Configuration ---
BASE_URL = "https://www.federalreserve.gov"
# The Federal Reserve website organizes speeches by year.
START_YEAR = 2024
CURRENT_YEAR = 2025 # Changed to 2025 to reflect current time
YEAR_URLS = [f"{BASE_URL}/newsevents/speech/{year}-speeches.htm" for year in range(START_YEAR, CURRENT_YEAR + 1)]

def sanitize_filename(title):
    sanitized = re.sub(r'[\\/:*?"<>|]', '', title)
    sanitized = sanitized.replace(' ', '_')
    if len(sanitized) > 100:
        sanitized = sanitized[:100] + "..."
    return sanitized.strip()

async def get_speech_details(page, speech_url):
    """
    Navigates to an individual speech page, extracts its title and text.
    Returns a tuple (title, text, url) or (None, None, url) on error.
    """
    try:
        await page.goto(speech_url, wait_until="domcontentloaded", timeout=60000) # Increased timeout

        main_content_css_selectors = [
            'div#article',                               # New: Found 'id="article"'
            'div.col-xs-12.col-sm-8.col-md-8',           # New: Responsive column class
            'div.col-md-8.col-md-push-2',                # Original (keep as fallback for older pages)
            'div.frb-text-content',                      # Another common FRB pattern
            'div[role="main"]',                          # Main role
            'article'                                    # General article tag
        ]
        
        # Join them with a comma and a space for clarity in the CSS selector string
        combined_css_selector = ', '.join(main_content_css_selectors)

        # Playwright will wait for ANY of these selectors to become visible
        await page.wait_for_selector(combined_css_selector, state='visible', timeout=30000)
        
        html_content = await page.content() # Get the fully rendered HTML
        soup = BeautifulSoup(html_content, 'html.parser')

        # Find the content div using BeautifulSoup, trying the selectors in order of preference
        content_div = None
        for selector in main_content_css_selectors:
            if '#' in selector: # e.g., 'div#article'
                tag_name, element_id = selector.split('#', 1) # Split only on the first '#'
                content_div = soup.find(tag_name, id=element_id)
            elif '.' in selector: # e.g., 'div.col-xs-12' or 'div.col-md-8.col-md-push-2'
                parts = selector.split('.')
                tag_name = parts[0] if parts[0] else None # 'div' or '' if starts with .
                class_names = [cls for cls in parts[1:] if cls] # Get all classes after tag
                
                # If no tag name specified (e.g., '.myclass'), default to any tag
                if not tag_name:
                    content_div = soup.find(lambda tag: tag.name is not None and all(cls in tag.get('class', []) for cls in class_names))
                else:
                    # Find all elements of the tag name and then filter by classes
                    possible_divs = soup.find_all(tag_name)
                    for div in possible_divs:
                        if all(cls in div.get('class', []) for cls in class_names):
                            content_div = div
                            break # Found the first matching one
            else: # Just a tag name like 'article'
                content_div = soup.find(selector)
            
            if content_div:
                # print(f"DEBUG: Found content_div using selector: {selector}") # Debugging line
                break # Found the most preferred content div, stop searching

        # Extract Speech Title
        speech_title = ""
        title_tag = soup.find('h3', class_='title')
        if title_tag:
            speech_title = title_tag.get_text(strip=True)
        else:
            # Fallback: Sometimes the title is in an H1 or H2
            # Check the hierarchy: sometimes it's under 'div.heading'
            heading_div = soup.find('div', class_='heading')
            if heading_div:
                title_tag_alt = heading_div.find('h1') or heading_div.find('h2')
                if title_tag_alt:
                    speech_title = title_tag_alt.get_text(strip=True)
            if not speech_title: # If still no title found
                title_tag_generic = soup.find('h1') or soup.find('h2')
                if title_tag_generic:
                    speech_title = title_tag_generic.get_text(strip=True)
                else:
                    speech_title = "Untitled Speech"
            print(f"Warning: Title fallback used for {speech_url}. Title: '{speech_title}'")


        # Extract Speech Text
        speech_text = ""
        if content_div:
            # Assuming paragraphs are direct children or within a few levels
            # might need to adjust this if text is in different tags (e.g., <div> with no <p> tags)
            paragraphs = content_div.find_all('p')
            # for p_tag in paragraphs:
            #     # Inside this loop, 'p_tag' is a single Tag object,
            #     # so can call find_all(), find(), get_text(), etc., on it.
            #     for br in p_tag.find_all('br'):
            #         br.replace_with(". ") 
            if not paragraphs: # If no <p> tags found, try extracting text directly from the div
                speech_text = content_div.get_text('\n\n', strip=True) # Join with double newline
            else:
                speech_text = '\n\n'.join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
            
            # Additional cleanup for common artifacts (e.g., footnotes, introductory lines)
            lines = speech_text.split('\n')
            cleaned_lines = [line for line in lines if not any(kw in line for kw in ["For release on delivery", "Board of Governors of the Federal Reserve System", "For release on:", "For release at:"])]
            speech_text = '\n'.join(cleaned_lines).strip()
            
        else:
            print(f"Warning: Could not find ANY main content div for {speech_url} after all attempts.")
            return speech_title, "", speech_url # Return title, empty text, and URL

        return speech_title, speech_text, speech_url

    except Exception as e:
        print(f"Error extracting details from {speech_url}: {e}")
        return None, None, speech_url # Return Nones on error

async def main():
    all_speech_data = [] # List to hold dictionaries for the DataFrame

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True) # Set headless=False for visual debugging
        page = await browser.new_page()

        # Iterate through each year's speeches page
        for year_url in YEAR_URLS:
            print(f"\n--- Processing: {year_url} ---")
            try:
                await page.goto(year_url, wait_until="domcontentloaded", timeout=60000)
                
                # Wait for the speech list containers to load
                # The 'eventlist__event' class is found inside 'col-xs-9 col-md-10' on list pages
                await page.wait_for_selector('div.eventlist__event', timeout=30000)

                html_content = await page.content() # Get the fully rendered HTML of the list page
                soup = BeautifulSoup(html_content, 'html.parser')

                speech_links_container = soup.find_all('div', class_='eventlist__event')

                if not speech_links_container:
                    print(f"No speech links containers found for {year_url}. Check selector or if content is truly dynamic for this specific page.")
                    continue

                current_year_speech_urls = []
                for container in speech_links_container:
                    link_tag = container.find('a', href=True)
                    # Also try to extract the date and speaker from the listing page if possible
                    speaker_tag = container.find('p', class_='news__speaker')

                    if link_tag and 'speech' in link_tag['href']:
                        relative_url = link_tag['href']
                        full_url = requests.compat.urljoin(BASE_URL, relative_url)
                        
                        # Store metadata from the list page
                        speech_meta = {
                            "link": full_url,
                            "speaker": speaker_tag.get_text(strip=True) if speaker_tag else None,
                            "title_from_list": link_tag.get_text(strip=True) # Title from the list link
                        }
                        current_year_speech_urls.append(speech_meta)

                print(f"Found {len(current_year_speech_urls)} potential speech links for {year_url.split('/')[-1]}.")

                for i, speech_meta in enumerate(current_year_speech_urls):
                    url = speech_meta["link"]
                    print(f"  [{i+1}/{len(current_year_speech_urls)}] Extracting: {url}")
                    
                    speech_title, speech_text, _ = await get_speech_details(page, url)
                    
                    # Add extracted data to the current speech's metadata
                    speech_meta["speech_title"] = speech_title
                    speech_meta["speech_text"] = speech_text
                    all_speech_data.append(speech_meta)
                    
                    # Delay a bit
                    time.sleep(0.5) 

            except Exception as e:
                print(f"Error processing year page {year_url}: {e}")
                continue # Move to the next year if an error occurs

        await browser.close()

    # Create the DataFrame
    df = pd.DataFrame(all_speech_data)
    
    # Reorder columns for clarity
    df = df[['speaker', 'speech_title', 'speech_text', 'link', 'title_from_list']]
    
    # Print basic info about the DataFrame
    print("\n--- Scraping Complete ---")
    print(f"Total speeches extracted: {len(df)}")
    print("\nDataFrame Head:")
    print(df.head())
    print("\nDataFrame Info:")
    df.info()
    
    # Save the DataFrame to a CSV file
    df.to_csv("fed_speeches_dataframe.csv", index=False, encoding='utf-8')
    print("\nDataFrame saved to fed_speeches_dataframe.csv")

await main()


--- Processing: https://www.federalreserve.gov/newsevents/speech/2024-speeches.htm ---
Found 104 potential speech links for 2024-speeches.htm.
  [1/104] Extracting: https://www.federalreserve.gov/newsevents/speech/kugler20241203a.htm
  [2/104] Extracting: https://www.federalreserve.gov/newsevents/speech/waller20241202a.htm
  [3/104] Extracting: https://www.federalreserve.gov/newsevents/speech/bowman20241122a.htm
  [4/104] Extracting: https://www.federalreserve.gov/newsevents/speech/bowman20241120a.htm
  [5/104] Extracting: https://www.federalreserve.gov/newsevents/speech/cook20241120a.htm
  [6/104] Extracting: https://www.federalreserve.gov/newsevents/speech/powell20241114a.htm
  [7/104] Extracting: https://www.federalreserve.gov/newsevents/speech/kugler20241114a.htm
  [8/104] Extracting: https://www.federalreserve.gov/newsevents/speech/waller20241112a.htm
  [9/104] Extracting: https://www.federalreserve.gov/newsevents/speech/bowman20241023a.htm
  [10/104] Extracting: https://www.fede

# Import the data and adjust some stuffs

In [55]:
df = pd.read_csv("fed_speeches_dataframe.csv")
print(df.shape)
df.head()

(223, 5)


,speaker,speech_title,speech_text,link,title_from_list
0,Governor Adriana D. Kugler,A Year in Review: A Tale of Two Supply Shocks,"December 03, 2024\n\nGovernor Adriana D. Kugle...",https://www.federalreserve.gov/newsevents/spee...,A Year in Review: A Tale of Two Supply Shocks
1,Governor Christopher J. Waller,Cut or Skip?,"December 02, 2024\n\nGovernor Christopher J. W...",https://www.federalreserve.gov/newsevents/spee...,Cut or Skip?
2,Governor Michelle W. Bowman,Artificial Intelligence in the Financial System,"November 22, 2024\n\nGovernor Michelle W. Bowm...",https://www.federalreserve.gov/newsevents/spee...,Artificial Intelligence in the Financial System
3,Governor Michelle W. Bowman,Approaching Policymaking Pragmatically,"November 20, 2024\n\nGovernor Michelle W. Bowm...",https://www.federalreserve.gov/newsevents/spee...,Approaching Policymaking Pragmatically
4,Governor Lisa D. Cook,Economic Outlook,"November 20, 2024\n\nGovernor Lisa D. Cook\n\n...",https://www.federalreserve.gov/newsevents/spee...,Economic Outlook


In [57]:
split_data = df['speech_text'].str.split('\n\n', n=3, expand=True)
# Assign the split parts to new columns
df['date_str'] = split_data[0]
df['speaker_2'] = split_data[1]
df['location'] = split_data[2]
df['speech_content'] = split_data[3]
df.head()


,speaker,speech_title,speech_text,link,title_from_list,date_str,speaker_2,location,speech_content
0,Governor Adriana D. Kugler,A Year in Review: A Tale of Two Supply Shocks,"December 03, 2024\n\nGovernor Adriana D. Kugle...",https://www.federalreserve.gov/newsevents/spee...,A Year in Review: A Tale of Two Supply Shocks,"December 03, 2024",Governor Adriana D. Kugler,"At the Detroit Economic Club, Detroit, Michigan","Thank you, Jason, and thank you for the opport..."
1,Governor Christopher J. Waller,Cut or Skip?,"December 02, 2024\n\nGovernor Christopher J. W...",https://www.federalreserve.gov/newsevents/spee...,Cut or Skip?,"December 02, 2024",Governor Christopher J. Waller,"At ""Building a Better Fed Framework,"" American...","Thank you, Lydia, and thank you for the opport..."
2,Governor Michelle W. Bowman,Artificial Intelligence in the Financial System,"November 22, 2024\n\nGovernor Michelle W. Bowm...",https://www.federalreserve.gov/newsevents/spee...,Artificial Intelligence in the Financial System,"November 22, 2024",Governor Michelle W. Bowman,At the 27th Annual Symposium on Building the F...,Discussions of artificial intelligence (AI) in...
3,Governor Michelle W. Bowman,Approaching Policymaking Pragmatically,"November 20, 2024\n\nGovernor Michelle W. Bowm...",https://www.federalreserve.gov/newsevents/spee...,Approaching Policymaking Pragmatically,"November 20, 2024",Governor Michelle W. Bowman,"At the Forum Club of the Palm Beaches, West Pa...",Good afternoon.1It is a pleasure to join you f...
4,Governor Lisa D. Cook,Economic Outlook,"November 20, 2024\n\nGovernor Lisa D. Cook\n\n...",https://www.federalreserve.gov/newsevents/spee...,Economic Outlook,"November 20, 2024",Governor Lisa D. Cook,"At the University of Virginia, Charlottesville...","Thank you, Christa. It is wonderful to be with..."


In [59]:
# transform date time
# first make sure the number of variable not nat:
print(df.shape)
print(df.isnull().sum())
# now transform
df['date'] = pd.to_datetime(df['date_str'], errors='coerce')
# check again
print(df.shape)
print(df.isnull().sum())

(223, 9)
speaker            0
speech_title       0
speech_text        0
link               0
title_from_list    0
date_str           0
speaker_2          0
location           0
speech_content     0
dtype: int64
(223, 10)
speaker            0
speech_title       0
speech_text        0
link               0
title_from_list    0
date_str           0
speaker_2          0
location           0
speech_content     0
date               0
dtype: int64


In [61]:
df.head()

,speaker,speech_title,speech_text,link,title_from_list,date_str,speaker_2,location,speech_content,date
0,Governor Adriana D. Kugler,A Year in Review: A Tale of Two Supply Shocks,"December 03, 2024\n\nGovernor Adriana D. Kugle...",https://www.federalreserve.gov/newsevents/spee...,A Year in Review: A Tale of Two Supply Shocks,"December 03, 2024",Governor Adriana D. Kugler,"At the Detroit Economic Club, Detroit, Michigan","Thank you, Jason, and thank you for the opport...",2024-12-03
1,Governor Christopher J. Waller,Cut or Skip?,"December 02, 2024\n\nGovernor Christopher J. W...",https://www.federalreserve.gov/newsevents/spee...,Cut or Skip?,"December 02, 2024",Governor Christopher J. Waller,"At ""Building a Better Fed Framework,"" American...","Thank you, Lydia, and thank you for the opport...",2024-12-02
2,Governor Michelle W. Bowman,Artificial Intelligence in the Financial System,"November 22, 2024\n\nGovernor Michelle W. Bowm...",https://www.federalreserve.gov/newsevents/spee...,Artificial Intelligence in the Financial System,"November 22, 2024",Governor Michelle W. Bowman,At the 27th Annual Symposium on Building the F...,Discussions of artificial intelligence (AI) in...,2024-11-22
3,Governor Michelle W. Bowman,Approaching Policymaking Pragmatically,"November 20, 2024\n\nGovernor Michelle W. Bowm...",https://www.federalreserve.gov/newsevents/spee...,Approaching Policymaking Pragmatically,"November 20, 2024",Governor Michelle W. Bowman,"At the Forum Club of the Palm Beaches, West Pa...",Good afternoon.1It is a pleasure to join you f...,2024-11-20
4,Governor Lisa D. Cook,Economic Outlook,"November 20, 2024\n\nGovernor Lisa D. Cook\n\n...",https://www.federalreserve.gov/newsevents/spee...,Economic Outlook,"November 20, 2024",Governor Lisa D. Cook,"At the University of Virginia, Charlottesville...","Thank you, Christa. It is wonderful to be with...",2024-11-20


In [63]:
# Here checking if the columns are the same:
df['speakerCheck'] = (df['speaker'] == df['speaker_2'])
print(df['speakerCheck'].value_counts())

df['titleCheck'] = (df['speech_title'] == df['title_from_list'])
print(df['titleCheck'].value_counts())


speakerCheck
True    223
Name: count, dtype: int64
titleCheck
True    223
Name: count, dtype: int64


All good, dropping duplicate and rearrange data

In [66]:
df = df.drop(columns = ['speakerCheck','titleCheck','speech_text','speaker_2','title_from_list','date_str'])
df = df.rename(columns = {'speech_title':'title','speech_content':'content'})
print(df.columns.tolist())

['speaker', 'title', 'link', 'location', 'content', 'date']


In [68]:
df = df[['date','speaker','title','content','location','link']]

In [70]:
print(df.shape)
df.head()

(223, 6)


,date,speaker,title,content,location,link
0,2024-12-03,Governor Adriana D. Kugler,A Year in Review: A Tale of Two Supply Shocks,"Thank you, Jason, and thank you for the opport...","At the Detroit Economic Club, Detroit, Michigan",https://www.federalreserve.gov/newsevents/spee...
1,2024-12-02,Governor Christopher J. Waller,Cut or Skip?,"Thank you, Lydia, and thank you for the opport...","At ""Building a Better Fed Framework,"" American...",https://www.federalreserve.gov/newsevents/spee...
2,2024-11-22,Governor Michelle W. Bowman,Artificial Intelligence in the Financial System,Discussions of artificial intelligence (AI) in...,At the 27th Annual Symposium on Building the F...,https://www.federalreserve.gov/newsevents/spee...
3,2024-11-20,Governor Michelle W. Bowman,Approaching Policymaking Pragmatically,Good afternoon.1It is a pleasure to join you f...,"At the Forum Club of the Palm Beaches, West Pa...",https://www.federalreserve.gov/newsevents/spee...
4,2024-11-20,Governor Lisa D. Cook,Economic Outlook,"Thank you, Christa. It is wonderful to be with...","At the University of Virginia, Charlottesville...",https://www.federalreserve.gov/newsevents/spee...


In [72]:
# remove some unnecessary words:
df['content'] = df['content'].str.replace(r'Return to text', '', regex=True)
print(df['content'].iloc[0])

Thank you, Jason, and thank you for the opportunity to speak here in Detroit today.1This visit has allowed me to see the many encouraging signs in this region: growth here in downtown; the area's famously hard-working labor force; and the gritty Detroit Lions, with 11 wins this season and counting. The nation has taken notice. But, truly, one of my favorite parts of serving as a Governor on the Federal Reserve Board is visiting communities across the country and hearing directly from the families, workers, and businesses we serve.

I am also glad to have the chance to speak with you near the end of the year. I think it is an appropriate time to look back and assess how the U.S. economy has developed over the course of 2024. I will also share with you my outlook and offer my views on U.S. monetary policy.

I will start by saying that I view the economy as being in a good position after making significant progress in recent years toward our dual-mandate goals of maximum employment and st

In [74]:
def separate_stuck_words(df, column_name):
    """
    Separates words that are stuck together (e.g., "2024Entering", "EconomyEntering")
    by inserting a space before an uppercase letter that follows a lowercase letter or a digit.
    Also prints all instances of such "violations" found before correction.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to clean.

    Returns:
        pd.DataFrame: The DataFrame with the specified column cleaned.
    """
    # Regex pattern:
    # (lookbehind)      (?<=[a-z\d])  - asserts that the match is preceded by a lowercase letter or a digit
    # (match)           ([A-Z])       - captures an uppercase letter
    # Replacement: ' \1' - inserts a space followed by the captured uppercase letter.
    # This effectively puts a space *before* the uppercase letter without removing the preceding char.
    pattern = r'(?<=[a-z\d])([A-Z])'

    # Apply the regex replacement to the specified column
    df[column_name] = df[column_name].str.replace(pattern, r' \1', regex=True)

    return df


In [76]:
dfFinish = separate_stuck_words(df.copy(), 'content')
print(dfFinish['content'].iloc[0])


Thank you, Jason, and thank you for the opportunity to speak here in Detroit today.1 This visit has allowed me to see the many encouraging signs in this region: growth here in downtown; the area's famously hard-working labor force; and the gritty Detroit Lions, with 11 wins this season and counting. The nation has taken notice. But, truly, one of my favorite parts of serving as a Governor on the Federal Reserve Board is visiting communities across the country and hearing directly from the families, workers, and businesses we serve.

I am also glad to have the chance to speak with you near the end of the year. I think it is an appropriate time to look back and assess how the U.S. economy has developed over the course of 2024. I will also share with you my outlook and offer my views on U.S. monetary policy.

I will start by saying that I view the economy as being in a good position after making significant progress in recent years toward our dual-mandate goals of maximum employment and s

In [78]:
print(dfFinish.shape)
dfFinish.head()


(223, 6)


,date,speaker,title,content,location,link
0,2024-12-03,Governor Adriana D. Kugler,A Year in Review: A Tale of Two Supply Shocks,"Thank you, Jason, and thank you for the opport...","At the Detroit Economic Club, Detroit, Michigan",https://www.federalreserve.gov/newsevents/spee...
1,2024-12-02,Governor Christopher J. Waller,Cut or Skip?,"Thank you, Lydia, and thank you for the opport...","At ""Building a Better Fed Framework,"" American...",https://www.federalreserve.gov/newsevents/spee...
2,2024-11-22,Governor Michelle W. Bowman,Artificial Intelligence in the Financial System,Discussions of artificial intelligence (AI) in...,At the 27th Annual Symposium on Building the F...,https://www.federalreserve.gov/newsevents/spee...
3,2024-11-20,Governor Michelle W. Bowman,Approaching Policymaking Pragmatically,Good afternoon.1 It is a pleasure to join you ...,"At the Forum Club of the Palm Beaches, West Pa...",https://www.federalreserve.gov/newsevents/spee...
4,2024-11-20,Governor Lisa D. Cook,Economic Outlook,"Thank you, Christa. It is wonderful to be with...","At the University of Virginia, Charlottesville...",https://www.federalreserve.gov/newsevents/spee...


In [80]:
df.to_csv('FedSpeeches.csv', date_format='%Y-%m-%d', index = False)


# Extracting transcripts from Youtube video - case by case 
(cannot do for long video)

In [3]:
import os
import re
from youtube_transcript_api import YouTubeTranscriptApi
import json
import asyncio
import google.generativeai as genai # Import the Gemini library

In [5]:
def extract_video_id(url):
    """
    Extract the video ID from a YouTube URL.
    
    Handles URLs like:
    - https://www.youtube.com/watch?v=VIDEO_ID
    - https://youtu.be/VIDEO_ID
    """
    # Pattern for full YouTube URL
    match = re.search(r"(?:v=)([^&#]+)", url)
    if match:
        return match.group(1)
    # Pattern for shortened URL
    match = re.search(r"(?:youtu\.be/)([^&#]+)", url)
    if match:
        return match.group(1)
    return None

def download_transcript(url, time = False, start_time = None, end_time = None):
    '''
    time: boolean, true if only getting part of it
    start_time, end_time: list of 3 elements [hours, minutes, second]
    '''
    video_id = extract_video_id(url)
    if not video_id:
        print("Error: Could not extract video ID from the URL provided.")
        return

    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id)
    except Exception as e:
        print(f"Error: Could not download transcript. Details: {e}")
        return

    # If only using parts of the video:
    if time:
        start = 3600*start_time[0]+60*start_time[1]+start_time[2]
        end = 3600*end_time[0]+60*end_time[1]+end_time[2]
        filtered_transcript = [entry for entry in transcript 
                               if start <= entry['start'] <= end]
    else:
        filtered_transcript = transcript
    
    # Print the transcript text line by line
    print("numbers of lines: " + str(len(filtered_transcript)))
    transcriptList = []
    for entry in filtered_transcript:
        transcriptList.append(entry["text"])    
    return transcriptList

def save_string_to_file(filename, content):
  """
  Saves a given string content to a specified text file.

  Args:
    filename: The name of the file to save to (e.g., "output.txt").
    content: The string content to write to the file.
  """
  try:
    with open(filename, 'w', encoding='utf-8') as file:
      file.write(content)
    print(f"Successfully saved content to '{filename}'")
  except IOError as e:
    print(f"Error saving file '{filename}': {e}")



In [7]:
# if link has live/ then replace it with watch?v=
link = 'https://www.youtube.com/watch?v=oKwXGG3tSh8'
transcriptList = download_transcript(link)
transcriptFull = ' '.join(transcriptList)
print(transcriptFull[:1000])
normalizedtranscript = re.sub(r'\s+', ' ', transcriptFull)
print(normalizedtranscript[:1000])


numbers of lines: 1354
Good afternoon. Hi. Good afternoon, everyone. Uh we're going to start our formal program now. I'm going to I've got the pleasure of introducing our guest speaker today. My name is Neil Kashkari. Uh I'm going to introduce our guest speaker, Kristoff Beck. Kristoff is chairman and chief executive officer of EcoAB, the global leader in water hygiene and infection prevention solutions. Uh, EcoAB has about 48,000 employees scattered all around the world. So, it's a big globally relevant company. Uh, I know Kristoff is passionate about water stewardship and the environment. We're going to get into that and the really important role that EcoAB plays in those uh, important initiatives. Kristoff was named Eolab's president and chief executive officer uh, in January of 2021 and he became chairman in May of 2022. He joined Eolab in 2007. Uh he's got a really remarkable background which I'm not going to get into in detail so we leave time for him. He's a native of Switzerlan

In [11]:
# if link has live/ then replace it with watch?v=
link = 'https://www.youtube.com/watch?v=rCjw93sODzY'
transcriptList = download_transcript(link, time=True, start_time = [0,8,25],end_time=[0,53,22])
transcriptFull = ' '.join(transcriptList)
print(transcriptFull[:1000])
normalizedtranscript = re.sub(r'\s+', ' ', transcriptFull)
print(normalizedtranscript[:1000])


numbers of lines: 1168
All right. So, uh, thank you, Abby, for the the warm introduction. This actually this this appearance is actually years in the making. I think the first invitation came like two and a half years ago and trying to get calendars and things together uh was difficult but I'm really glad to be here. So uh have some prepared remarks and then we I'm looking forward to the conversation. So good morning and thanks to the Federal Reserve Bank of Minneapolis and especially the team at the opportunity and inclusive growth institute for putting this conference together and for inviting me to participate. As I said, it's great to be here and um I' I'm really excited to be talking about these issues and to uh get to see the community that is working on this stuff in the system. The pursuit of inclusive economic growth is at the center of much of my work. Uh and it's a soapbox issue for me at a personal level. Economic inclusion has been a theme from my earliest days in the econ

In [9]:
# if there is no need to add periods or comma
save_string_to_file('processed.txt', normalizedtranscript)


Successfully saved content to 'processed.txt'


## Youtube transcript adding periods or comma
If the above does not have comma or periods, start using the below

In [14]:
from deepmultilingualpunctuation import PunctuationModel
model = PunctuationModel()


Device set to use cpu
C:\Users\Lynn\anaconda3\Lib\site-packages\transformers\pipelines\token_classification.py:170: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.NONE"` instead.
  warnings.warn(


In [17]:
def save_string_to_file(filename, content):
  """
  Saves a given string content to a specified text file.

  Args:
    filename: The name of the file to save to (e.g., "output.txt").
    content: The string content to write to the file.
  """
  try:
    with open(filename, 'w', encoding='utf-8') as file:
      file.write(content)
    print(f"Successfully saved content to '{filename}'")
  except IOError as e:
    print(f"Error saving file '{filename}': {e}")



In [17]:
result = model.restore_punctuation(normalizedtranscript)
print(result[:1000])


All right, good morning everyone, and thank you so much for being here with us today. We're really excited for the um, for the program to kick off here. I'm going to keep my opening comments very brief as we begin, but, uh, just thank you for making the time. We're very excited about this program and and grateful for the Federal Reserve team for making it happen. Uh, for those of you that I haven't had the chance to meet, my name is Dustin Taylor and I proudly serve as the executive director with the Florida Institute of CFOs, CFO Exchange Group and Chief People Group. On behalf of our 450 members around the state of Florida, we'd like to extend our sincere thank you to Dr Bostik and his team for joining us at such a pivotal time in our economy. It's truly an honor to introduce today's distinguished speakers. Dr Bostik is the president and CEO of the Federal Reserve Bank of Atlanta and serves on the Federal Open Market Committee, where he votes every third year. Before moving to the so

In [18]:
save_string_to_file('processed.txt', result)


Successfully saved content to 'processed.txt'


# Importing text files for Bank Presidents

In [10]:
# import packages
import numpy as np
np.random.seed(4)
import matplotlib.pyplot as plt
import seaborn as sns

from itertools import groupby
import pandas as pd

import os
import re
import string

import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm as tq
tq.pandas() #thanks to https://stackoverflow.com/questions/18603270/progress-indicator-during-pandas-operations

In [84]:
def extractFedSpeeches():
    CBList = ['Boston','NewYork','Philadelphia','Cleveland','Richmond',
              'Atlanta','Chicago','StLouis','Minneapolis','Kansas',
              'Dallas','SanFrancisco']
    CBElementList = [['link','title','speaker','date','location'],
                     ['link','title','date','speaker','location'],
                     ['link','title','speaker','date','location'],
                     ['link','title','speaker','date','location'],
                     ['link','title','date','speaker','location'],
                     ['link','title','speaker','location','date'],
                     ['link','date','title','speaker','location'],
                     ['link','title','date','speaker','location'],
                     ['link','title','date','location','speaker'],
                     ['link','title','location','date','speaker'],
                     ['link','speaker','title','date','location'],
                     ['link','title','speaker','date','location']
                    ]
    generalDir = 'RawDataNew/'
    varList1 = ['District','text','MultSpeakers','VideoForm','Article']
    varList2 = varList1 + ['speechContent','title','date','speaker','location','link']
    df = pd.DataFrame(columns= varList2)
    
    for CB,elements in zip(CBList,CBElementList):
        print('Importing CB of {}'.format(CB))
        cbDir = generalDir + '{}/'.format(CB)
        if os.path.exists(cbDir):
            for filename in os.listdir(cbDir):
                # get the directory, also check validity
                # for some reason all the file name has the "._" in front of their name, hence there is a need to adjust
                print(filename)
                path = os.path.join(cbDir, filename)
                if 'DS_Store' in path:
                    print('Invalid files')
                    continue
                    
                #See if "Mult","Art" or "Vid" is in file name
                mult = 'Mult' in filename
                vid = 'Vid' in filename
                art = 'Art' in filename

                #get the text
                try:
                    with open(path, 'r', encoding="utf8") as path:
                        text = path.read()
                except:
                    print(path)
                    print(filename)
                    continue
                
                #combine
                dum = pd.DataFrame([[CB,text,mult,vid,art]], columns = varList1)
                
                # Now separate the text into more columns depending on the CB
                split_data = dum['text'].str.split('\n\n', n=len(elements), expand=True)
                # Assign the split parts to new columns
                for i in range(len(elements)):
                    dum[elements[i]] = split_data[i]
                dum['speechContent'] = split_data[len(elements)]
                
                # Rearrange and combine to the main df
                dum = dum[varList2]
                df = pd.concat([df, dum])
        
    return df


In [86]:
df = extractFedSpeeches()

Importing CB of Boston
1.txt
10.txt
11.txt
12.txt
13_Vid_Mult.txt
14.txt
15.txt
16.txt
17.txt
18.txt
19.txt
2.txt
20.txt
21.txt
22.txt
23.txt
24.txt
3_Mult.txt
4_Mult.txt
5.txt
6.txt
7_Mult.txt
8.txt
9.txt
Importing CB of NewYork
1.txt
10.txt
11.txt
12.txt
13.txt
14.txt
15.txt
16.txt
17.txt
18.txt
19.txt
2.txt
20.txt
21.txt
22.txt
23.txt
24.txt
25.txt
26.txt
27.txt
28.txt
29.txt
3.txt
30.txt
31.txt
32.txt
33.txt
34.txt
35_Mult.txt
36.txt
37.txt
38.txt
39.txt
4.txt
40.txt
41_Mult.txt
42.txt
43.txt
44.txt
45.txt
46.txt
47.txt
48.txt
49.txt
5.txt
50.txt
51.txt
52.txt
53.txt
54.txt
6.txt
7.txt
8.txt
9.txt
Importing CB of Philadelphia
1.txt
10.txt
11.txt
12.txt
13.txt
14.txt
15.txt
16.txt
17.txt
18.txt
19.txt
2.txt
3.txt
4.txt
5.txt
6.txt
7.txt
8.txt
9.txt
Importing CB of Cleveland
1.txt
10.txt
11.txt
12.txt
13.txt
14.txt
15.txt
16.txt
17.txt
18.txt
2.txt
3.txt
4.txt
5.txt
6.txt
7.txt
8.txt
9.txt
Importing CB of Richmond
1.txt
10.txt
11.txt
12.txt
13.txt
14.txt
15.txt
16.txt
17.txt
18.txt
2

In [87]:
print(df.shape)
print(df.columns.tolist())

(278, 11)
['District', 'text', 'MultSpeakers', 'VideoForm', 'Article', 'speechContent', 'title', 'date', 'speaker', 'location', 'link']


In [88]:
# briefly checking for each central bank are they having the correct value for each variables?
CBList = ['Boston','NewYork','Philadelphia','Cleveland','Richmond',
              'Atlanta','Chicago','StLouis','Minneapolis','Kansas',
              'Dallas','SanFrancisco']

for CB in CBList:
    print('For '+CB)
    print(df[df['District'] == CB].shape)
    print(df[df['District'] == CB].head())
    print('#'*100)



For Boston
(24, 11)
  District                                               text MultSpeakers  \
0   Boston  https://www.bostonfed.org/news-and-events/spee...        False   
0   Boston  https://www.bostonfed.org/news-and-events/spee...        False   
0   Boston  https://www.bostonfed.org/news-and-events/spee...        False   
0   Boston  https://www.bostonfed.org/news-and-events/spee...        False   
0   Boston  https://www.bostonfed.org/news-and-events/spee...         True   

  VideoForm Article                                      speechContent  \
0     False   False  Economic Resilience, Amid Elevated Tariffs and...   
0     False   False  I am pleased to welcome all of you to the Fede...   
0     False   False  Takeaways from Boston Fed President Susan M. C...   
0     False   False  Boston Fed President and CEO Susan M. Collins'...   
0      True   False  hi, good afternoon everyone. my name is Beth B...   

                                               title              

In [89]:
# now transform
df['dateFormat'] = pd.to_datetime(df['date'], errors='coerce')
# check again
print(df.shape)
print(df.isnull().sum())

(278, 12)
District          0
text              0
MultSpeakers      0
VideoForm         0
Article           0
speechContent     0
title             0
date              0
speaker           0
location          0
link              0
dateFormat       81
dtype: int64


In [90]:
# empty date is because the format is not recognized, and it is multiple countries
from datetime import datetime

# List of known formats
known_formats = [
    "%d %b %Y",      # 05 Jun 2025
    "%m.%d.%Y",      # 06.26.2025
    "%b. %d, %Y",    # Nov. 12, 2024
    "%b %d, %Y",     # Oct 18, 2024
    "%m/%d/%y",      # 02/28/25
    "%Y-%m-%d",      # 2025-06-05
    "%Y/%m/%d",      # 2025/06/05
    "%b %d %Y",      # Sep 27 2024
    "%B %d, %Y"      # April 23, 2025
]

def parse_dates(date_str):
    for fmt in known_formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return pd.NaT  # Return Not a Time if all formats fail

# Apply to your DataFrame
mask = df['dateFormat'].isna()
df.loc[mask,'dateFormat'] = df.loc[mask,'date'].apply(parse_dates)


In [91]:
df[df['dateFormat'].isnull()][['District','dateFormat','date']]

,District,dateFormat,date
0,Richmond,NaT,"Sept. 26, 2025"
0,StLouis,NaT,"Sept. 3, 2025"
0,StLouis,NaT,"Sept. 22, 2025"
0,Kansas,NaT,"November 13, 2024"
0,Kansas,NaT,"November 19, 2024"
0,Kansas,NaT,January 9. 2025
0,Kansas,NaT,"January 14, 2025"
0,Kansas,NaT,"February 26, 2024"


In [92]:
# rename and drop date
df = df.drop(columns = 'date')
df = df.rename(columns = {'dateFormat':'date'})

In [93]:
# Here remove some texts
def remove_hyperlinks(text):
    url_pattern = r'https?://\S+|www\.\S+'
    return re.sub(url_pattern, '', text)
    
# remove some unnecessary words:
df['speechContent'] = df['speechContent'].str.replace(r'Return to \d+', '', regex=True)
# remove hyperlinks
df['speechContent'] = df['speechContent'].apply(remove_hyperlinks)


In [94]:
# check for if it is removed
dfcheck = df[df['title'] == 'Community Development: Paying It Forward'].copy()
print(dfcheck.at[0,'speechContent'])



Introduction
Good morning and thank you all for joining us at the seventeenth Policy Summit. I’m honored to welcome you to our marquee community development event—my first as president and CEO of the Federal Reserve Bank of Cleveland.1

If this is your first visit to our fair city, let me assure you Cleveland does indeed rock. I highly recommend exploring some of what this vibrant city has to offer in one of our loveliest weather months. The Rock and Roll Hall of Fame is just down the street, as is Progressive Field, where I happen to know the Guardians are playing home games over the next couple of days.

But while we are participating in the summit, I’d like to acknowledge we wouldn’t all be together in person, or virtually, if it wasn’t for the efforts of the Cleveland Fed’s community development staff and our Reserve Bank and community partners. The Federal Reserve System’s Community Development Departments aim to promote economic growth and financial stability. We are honored to w

# Combine with Governors, and fix format

In [104]:
print(df.columns.tolist())
print(df.shape)

df2 = pd.read_csv('FedSpeeches.csv', parse_dates = ['date'])
print(df2.columns.tolist())
print(df2.shape)


['District', 'text', 'MultSpeakers', 'VideoForm', 'Article', 'speechContent', 'title', 'speaker', 'location', 'link', 'date']
(278, 11)
['date', 'speaker', 'title', 'content', 'location', 'link']
(223, 6)


In [106]:
# add missing columns for board governors
df2['District'] = 'BoardGovernors'
df2['MultSpeakers'] = False
df2['VideoForm'] = False
df2['Article'] = False

# fixing for bpres
df = df.rename(columns = {'speechContent':'content'})
df = df.drop(columns = ['text'])

# check again
print(df.columns.tolist())
print(df.shape)

print(df2.columns.tolist())
print(df2.shape)


['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date']
(278, 10)
['date', 'speaker', 'title', 'content', 'location', 'link', 'District', 'MultSpeakers', 'VideoForm', 'Article']
(223, 10)


In [108]:
# now merge and check
dfTotal = pd.concat([df, df2], ignore_index=True)
print(dfTotal.columns.tolist())
print(dfTotal.shape)


['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date']
(501, 10)


In [110]:
dfTotal.to_csv('FedSpeechesTotal.csv', date_format='%Y-%m-%d', index = False)


# Extend with Campiglioetal et al (2025) data

## Import data

In [12]:
df1 = pd.read_csv('FedSpeechesTotal.csv', parse_dates = ['date'])
print(df1.shape)
print(df1.columns.tolist())


(501, 10)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date']


In [14]:
df2 = pd.read_csv('Campiglioetal_2025/CBS_dataset_v1.0.csv', parse_dates = ['Date'])
print(df2.shape)
print(df2.columns.tolist())

dfus = df2[df2['Country'] == 'USA'].copy()
print(dfus.shape)
print(dfus['Date'].min())
print(dfus['Date'].max())
dfus.head()


(35487, 15)
['URL', 'PDF', 'Title', 'Subtitle', 'Date', 'Authorname', 'Role', 'Gender', 'CentralBank', 'Country', 'text', 'text_original', 'Filename', 'Language', 'Source']
(6607, 15)
1986-01-06 00:00:00
2023-12-11 00:00:00


,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source
12456,https://www.bis.org/review/r000114a.pdf,https://www.bis.org/review/r000114a.pdf,Mr Ferguson compares Asian and Latin American ...,"Remarks by Mr Roger W Ferguson Jr, Vice-Chairm...",2000-01-07,Roger W Ferguson,Deputy Governor,Male,Board of Governors of the Federal Reserve,USA,Mr Ferguson compares Asian and Latin American ...,NaN,r000114a,English,BIS
12457,https://www.bis.org/review/r000117a.pdf,https://www.bis.org/review/r000117a.pdf,Mr Greenspan discusses technology and the US e...,"Remarks by Mr Alan Greenspan, Chairman of the ...",2000-01-13,Alan Greenspan,Governor,Male,Board of Governors of the Federal Reserve,USA,Mr Greenspan discusses technology and the US e...,NaN,r000117a,English,BIS
12458,https://www.bis.org/review/r000117b.pdf,https://www.bis.org/review/r000117b.pdf,Mr Gramlich focuses on inflation targeting (C...,"Remarks by Mr Edward M Gramlich, Member of the...",2000-01-13,Edward M Gramlich,Board member,Male,Board of Governors of the Federal Reserve,USA,Mr Gramlich focuses on inflation targeting Rem...,NaN,r000117b,English,BIS
12461,https://www.bis.org/review/r000124b.pdf,https://www.bis.org/review/r000124b.pdf,Mr Meyer gives his views on the sustainability...,"Remarks by Mr Laurence H Meyer, Member of the ...",2000-01-20,Laurence H Meyer,Board member,Male,Board of Governors of the Federal Reserve,USA,Mr Meyer gives his views on the sustainability...,NaN,r000124b,English,BIS
12471,https://www.bis.org/review/r000216a.pdf,https://www.bis.org/review/r000216a.pdf,Mr Greenspan gives a testimony on over-the-cou...,"Testimony of Mr Alan Greenspan, Chairman of th...",2000-02-10,Alan Greenspan,Governor,Male,Board of Governors of the Federal Reserve,USA,Mr Greenspan gives a testimony on over-the-cou...,NaN,r000216a,English,BIS


## I need to check the missing values for St. Louis

In [16]:
dfus[dfus['CentralBank'] == 'Federal Reserve Bank of St Louis']['Authorname'].value_counts()


Authorname
James Bullard      163
Thomas C Melzer     89
Name: count, dtype: int64

Indeed that William Poole is missing, See the section "Data for William Poole later"

## Combining

In [115]:
# homogenize the CentralBank and District
print(df1['District'].value_counts().index.tolist())
print(dfus['CentralBank'].value_counts().index.tolist())

# Dictionary mapping old values to new ones
oldname = ['Board of Governors of the Federal Reserve', 'Federal Reserve Bank of New York', 'Federal Reserve Bank of Atlanta', 
           'Federal Reserve Bank of San Francisco', 'Federal Reserve Bank of Chicago', 'Federal Reserve Bank of Cleveland', 
           'Federal Reserve Bank of Philadelphia', 'Federal Reserve Bank of Richmond', 'Federal Reserve Bank of St Louis', 
           'Federal Reserve Bank of Boston', 'Federal Reserve Bank of Dallas', 'Federal Reserve Bank of Kansas City', 
           'Federal Reserve Bank of Minneapolis']
newname = ['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago', 'Cleveland', 'Philadelphia', 'Richmond',
           'StLouis', 'Boston', 'Dallas', 'Kansas','Minneapolis']

# Replace values in column A
dfus['District'] = dfus['CentralBank'].replace(dict(zip(oldname,newname)))

# checking
print(dfus['District'].value_counts().index.tolist())


['BoardGovernors', 'NewYork', 'StLouis', 'Minneapolis', 'Dallas', 'Boston', 'Philadelphia', 'Kansas', 'Cleveland', 'Richmond', 'Atlanta', 'Chicago', 'SanFrancisco']
['Board of Governors of the Federal Reserve', 'Federal Reserve Bank of New York', 'Federal Reserve Bank of Atlanta', 'Federal Reserve Bank of San Francisco', 'Federal Reserve Bank of Chicago', 'Federal Reserve Bank of Cleveland', 'Federal Reserve Bank of Philadelphia', 'Federal Reserve Bank of Richmond', 'Federal Reserve Bank of St Louis', 'Federal Reserve Bank of Boston', 'Federal Reserve Bank of Dallas', 'Federal Reserve Bank of Kansas City', 'Federal Reserve Bank of Minneapolis']
['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago', 'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Boston', 'Dallas', 'Kansas', 'Minneapolis']


In [118]:
# need to make the two combinable with columns
dfus['URL'] = dfus['URL'].fillna(dfus['PDF'])
dfus = dfus.rename(columns = {'URL':'link',
                              'Title':'title',
                              'Date':'date',
                              'Authorname':'speaker',
                              'text':'content'})
dfus['location'] = dfus['CentralBank']
dfus['MultSpeakers'] = np.nan # not sure because they only webscrapping and not really note down if there are multiple people
dfus['VideoForm'] = False
dfus['Article'] = False
dfusMerge = dfus[['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date']]

# now merge
print(df1.shape)
print(dfusMerge.shape)
dfTotal = pd.concat([df1, dfusMerge], ignore_index=True)


(501, 10)
(6607, 10)


In [120]:
print(dfTotal.shape)
dfTotal.head()

(7108, 10)


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16


In [122]:
dfTotal.to_csv('FedSpeechesTotalExtended.csv', date_format='%Y-%m-%d', index = False)


# Data for William Poole

## Get the basic data from the website without pdf content

In [63]:
url = "https://fraser.stlouisfed.org/title/statements-speeches-william-poole-485?browse=1990s"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Find the JSON-LD block
json_ld = soup.find("script", type="application/ld+json").string

data = json.loads(json_ld)

records = []

# The speeches are inside data["hasPart"]
for item in data["hasPart"]:
    title = item["name"]
    link = item["url"]
    records.append({"title": title, "url": link})

df = pd.DataFrame(records)
df.head()

,title,url
0,Economic Growth: Is the Fed Irrelevant? : St. ...,https://fraser.stlouisfed.org/title/statements...
1,A Policymaker Confronts Uncertainty : St. Loui...,https://fraser.stlouisfed.org/title/statements...
2,Is Inflation Too Low? : 16th Annual Monetary C...,https://fraser.stlouisfed.org/title/statements...
3,Whither the U.S. Credit Markets? : Constructio...,https://fraser.stlouisfed.org/title/statements...
4,That Mysterious FOMC : The Economic Club of Me...,https://fraser.stlouisfed.org/title/statements...
...,...,...
127,Dollars and Sense : Financial Planning Associa...,https://fraser.stlouisfed.org/title/statements...
128,Reflections : The St. Louis Gateway Chapter of...,https://fraser.stlouisfed.org/title/statements...
129,"Inflation Dynamics : The Baldwin Lecture, Trum...",https://fraser.stlouisfed.org/title/statements...
130,"Balancing Financial Stability, Price Stability...",https://fraser.stlouisfed.org/title/statements...


In [87]:
def extract_meta(soup, name):
    tag = soup.find("meta", attrs={"name": name})
    return tag["content"] if tag and tag.has_attr("content") else None

def extract_speech_details(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    return {
        "title": extract_meta(soup, "citation_title"),
        "author": extract_meta(soup, "citation_author"),
        "date": extract_meta(soup, "citation_date"),
        "pdf": extract_meta(soup, "citation_pdf_url"),
        "series": extract_meta(soup, "citation_series_title"),
        "type": extract_meta(soup, "citation_article_type"),
    }


In [89]:
details_list = []

for url in df["url"]:
    details = extract_speech_details(url)
    details_list.append(details)

details_df = pd.DataFrame(details_list)
df_full = pd.concat([df, details_df], axis=1)
df_full.head()

,title,url,title,author,date,pdf,series,type
0,Economic Growth: Is the Fed Irrelevant? : St. ...,https://fraser.stlouisfed.org/title/statements...,Economic Growth: Is the Fed Irrelevant? : St. ...,"Poole, William, 1937 June 19-",1998-07-15,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
1,A Policymaker Confronts Uncertainty : St. Loui...,https://fraser.stlouisfed.org/title/statements...,A Policymaker Confronts Uncertainty : St. Loui...,"Poole, William, 1937 June 19-",1998-09-16,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
2,Is Inflation Too Low? : 16th Annual Monetary C...,https://fraser.stlouisfed.org/title/statements...,Is Inflation Too Low? : 16th Annual Monetary C...,"Poole, William, 1937 June 19-",1998-10-22,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
3,Whither the U.S. Credit Markets? : Constructio...,https://fraser.stlouisfed.org/title/statements...,Whither the U.S. Credit Markets? : Constructio...,"Poole, William, 1937 June 19-",1998-10-26,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
4,That Mysterious FOMC : The Economic Club of Me...,https://fraser.stlouisfed.org/title/statements...,That Mysterious FOMC : The Economic Club of Me...,"Poole, William, 1937 June 19-",1998-12-03,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart


In [91]:
# temporary save before attempting to parse the pdf
df_full.to_csv('WilliamPooleNoContent.csv', date_format='%Y-%m-%d', index = False)


## Attempting to parse pdf file

In [98]:
from io import BytesIO
from pdfminer.high_level import extract_text

In [100]:
df = pd.read_csv("WilliamPooleNoContent.csv")
print(df.shape)
df.head()


(132, 8)


,title,url,title.1,author,date,pdf,series,type
0,Economic Growth: Is the Fed Irrelevant? : St. ...,https://fraser.stlouisfed.org/title/statements...,Economic Growth: Is the Fed Irrelevant? : St. ...,"Poole, William, 1937 June 19-",1998-07-15,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
1,A Policymaker Confronts Uncertainty : St. Loui...,https://fraser.stlouisfed.org/title/statements...,A Policymaker Confronts Uncertainty : St. Loui...,"Poole, William, 1937 June 19-",1998-09-16,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
2,Is Inflation Too Low? : 16th Annual Monetary C...,https://fraser.stlouisfed.org/title/statements...,Is Inflation Too Low? : 16th Annual Monetary C...,"Poole, William, 1937 June 19-",1998-10-22,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
3,Whither the U.S. Credit Markets? : Constructio...,https://fraser.stlouisfed.org/title/statements...,Whither the U.S. Credit Markets? : Constructio...,"Poole, William, 1937 June 19-",1998-10-26,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart
4,That Mysterious FOMC : The Economic Club of Me...,https://fraser.stlouisfed.org/title/statements...,That Mysterious FOMC : The Economic Club of Me...,"Poole, William, 1937 June 19-",1998-12-03,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart


In [102]:
def extract_pdf_text_from_url(pdf_url):
    response = requests.get(pdf_url)
    pdf_data = BytesIO(response.content)
    text = extract_text(pdf_data)
    return text

In [104]:
df["text"] = df["pdf"].apply(extract_pdf_text_from_url)
df.head()

,title,url,title.1,author,date,pdf,series,type,text
0,Economic Growth: Is the Fed Irrelevant? : St. ...,https://fraser.stlouisfed.org/title/statements...,Economic Growth: Is the Fed Irrelevant? : St. ...,"Poole, William, 1937 June 19-",1998-07-15,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Economic Growth: Is the Fed Irrelevant?\n\nSt....
1,A Policymaker Confronts Uncertainty : St. Loui...,https://fraser.stlouisfed.org/title/statements...,A Policymaker Confronts Uncertainty : St. Loui...,"Poole, William, 1937 June 19-",1998-09-16,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,A Policymaker Confronts Uncertainty\n\nSt. Lou...
2,Is Inflation Too Low? : 16th Annual Monetary C...,https://fraser.stlouisfed.org/title/statements...,Is Inflation Too Low? : 16th Annual Monetary C...,"Poole, William, 1937 June 19-",1998-10-22,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Is Inflation Too Low?\n\n16th Annual Monetary ...
3,Whither the U.S. Credit Markets? : Constructio...,https://fraser.stlouisfed.org/title/statements...,Whither the U.S. Credit Markets? : Constructio...,"Poole, William, 1937 June 19-",1998-10-26,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Whither the U.S. Credit Markets?\n\nConstructi...
4,That Mysterious FOMC : The Economic Club of Me...,https://fraser.stlouisfed.org/title/statements...,That Mysterious FOMC : The Economic Club of Me...,"Poole, William, 1937 June 19-",1998-12-03,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,That Mysterious FOMC\n\nThe Economic Club of M...


In [116]:
testing = df.loc[0,"text"]
testing[2000:5000]

'e city and had a very pleasant\nevening.\n\nBut as we drive around—and I’m sure you\nhave exactly the same reaction, if you just go pok-\ning around without any particular destination—\nyou do see some really serious urban problems\nin this area. In St. Louis and East St. Louis, there\nis a lot of what looked to me, driving by, to be\nabandoned industrial land. I’m not a development\nexpert or an urban economist, but you have to ask\nwhat can be done about these problems. St. Louis\nis not alone in having them, but there is no reason\nwhy St. Louis couldn’t be out ahead in fixing\nthem. I’m sure that would be the desire of every-\none in this room.\n\nDoes the Federal Reserve have a role in this\nprocess? Well, my main message is going to be\n“no,” but I want to talk to you about that and tell\nyou what the Federal Reserve can contribute. It’s\nvery important to understand what the function\nof the central bank is, what it can contribute and\nwhat it cannot, to understand where the re

In [118]:
# it works great, but there are a few formating problem which can be solved
def clean_pdf_text(text):
    # 1. Fix hyphenated line breaks
    text = re.sub(r'-\s*\n\s*', '', text)
    # 2. Replace remaining newlines with spaces
    text = re.sub(r'\s*\n\s*', ' ', text)
    # 3. Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["clean_text"] = df["text"].apply(clean_pdf_text)
print(df.loc[0, "clean_text"][2000:5000])


d a very pleasant evening. But as we drive around—and I’m sure you have exactly the same reaction, if you just go poking around without any particular destination— you do see some really serious urban problems in this area. In St. Louis and East St. Louis, there is a lot of what looked to me, driving by, to be abandoned industrial land. I’m not a development expert or an urban economist, but you have to ask what can be done about these problems. St. Louis is not alone in having them, but there is no reason why St. Louis couldn’t be out ahead in fixing them. I’m sure that would be the desire of everyone in this room. Does the Federal Reserve have a role in this process? Well, my main message is going to be “no,” but I want to talk to you about that and tell you what the Federal Reserve can contribute. It’s very important to understand what the function of the central bank is, what it can contribute and what it cannot, to understand where the responsibility really lies. I want to divide 

In [120]:
# save the full version and the picked out version
df.to_csv('WilliamPooleFull.csv', date_format='%Y-%m-%d', index = False)


## Try to make consistent with main, and run all the text analysis before merging

In [4]:
df = pd.read_csv('WilliamPooleFull.csv', parse_dates = ['date'])
print(df.shape)
df.head()


(132, 10)


,title,url,title.1,author,date,pdf,series,type,text,clean_text
0,Economic Growth: Is the Fed Irrelevant? : St. ...,https://fraser.stlouisfed.org/title/statements...,Economic Growth: Is the Fed Irrelevant? : St. ...,"Poole, William, 1937 June 19-",1998-07-15,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Economic Growth: Is the Fed Irrelevant?\n\nSt....,Economic Growth: Is the Fed Irrelevant? St. Lo...
1,A Policymaker Confronts Uncertainty : St. Loui...,https://fraser.stlouisfed.org/title/statements...,A Policymaker Confronts Uncertainty : St. Loui...,"Poole, William, 1937 June 19-",1998-09-16,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,A Policymaker Confronts Uncertainty\n\nSt. Lou...,A Policymaker Confronts Uncertainty St. Louis ...
2,Is Inflation Too Low? : 16th Annual Monetary C...,https://fraser.stlouisfed.org/title/statements...,Is Inflation Too Low? : 16th Annual Monetary C...,"Poole, William, 1937 June 19-",1998-10-22,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Is Inflation Too Low?\n\n16th Annual Monetary ...,Is Inflation Too Low? 16th Annual Monetary Con...
3,Whither the U.S. Credit Markets? : Constructio...,https://fraser.stlouisfed.org/title/statements...,Whither the U.S. Credit Markets? : Constructio...,"Poole, William, 1937 June 19-",1998-10-26,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Whither the U.S. Credit Markets?\n\nConstructi...,Whither the U.S. Credit Markets? Construction ...
4,That Mysterious FOMC : The Economic Club of Me...,https://fraser.stlouisfed.org/title/statements...,That Mysterious FOMC : The Economic Club of Me...,"Poole, William, 1937 June 19-",1998-12-03,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,That Mysterious FOMC\n\nThe Economic Club of M...,That Mysterious FOMC The Economic Club of Memp...


In [6]:
# Title includes the true title and the location
df[["title_clean", "location"]] = df["title"].str.split(" : ", n=1, expand=True)
df.head()

,title,url,title.1,author,date,pdf,series,type,text,clean_text,title_clean,location
0,Economic Growth: Is the Fed Irrelevant? : St. ...,https://fraser.stlouisfed.org/title/statements...,Economic Growth: Is the Fed Irrelevant? : St. ...,"Poole, William, 1937 June 19-",1998-07-15,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Economic Growth: Is the Fed Irrelevant?\n\nSt....,Economic Growth: Is the Fed Irrelevant? St. Lo...,Economic Growth: Is the Fed Irrelevant?,St. Louis Regional Commerce and Growth Associa...
1,A Policymaker Confronts Uncertainty : St. Loui...,https://fraser.stlouisfed.org/title/statements...,A Policymaker Confronts Uncertainty : St. Loui...,"Poole, William, 1937 June 19-",1998-09-16,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,A Policymaker Confronts Uncertainty\n\nSt. Lou...,A Policymaker Confronts Uncertainty St. Louis ...,A Policymaker Confronts Uncertainty,St. Louis Gateway Chapter of the National Asso...
2,Is Inflation Too Low? : 16th Annual Monetary C...,https://fraser.stlouisfed.org/title/statements...,Is Inflation Too Low? : 16th Annual Monetary C...,"Poole, William, 1937 June 19-",1998-10-22,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Is Inflation Too Low?\n\n16th Annual Monetary ...,Is Inflation Too Low? 16th Annual Monetary Con...,Is Inflation Too Low?,"16th Annual Monetary Conference, Cato Institut..."
3,Whither the U.S. Credit Markets? : Constructio...,https://fraser.stlouisfed.org/title/statements...,Whither the U.S. Credit Markets? : Constructio...,"Poole, William, 1937 June 19-",1998-10-26,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,Whither the U.S. Credit Markets?\n\nConstructi...,Whither the U.S. Credit Markets? Construction ...,Whither the U.S. Credit Markets?,"Construction Financial Managers Association, C..."
4,That Mysterious FOMC : The Economic Club of Me...,https://fraser.stlouisfed.org/title/statements...,That Mysterious FOMC : The Economic Club of Me...,"Poole, William, 1937 June 19-",1998-12-03,https://fraser.stlouisfed.org/files/docs/histo...,Statements and Speeches of William Poole,multipart,That Mysterious FOMC\n\nThe Economic Club of M...,That Mysterious FOMC The Economic Club of Memp...,That Mysterious FOMC,"The Economic Club of Memphis, Memphis, Tennessee"


In [8]:
df = df.drop(columns = ['title','title.1'])
df['District'] = "StLouis"
df['MultSpeakers'] = 0.0
df['VideoForm'] = False
df['Article'] = False
df = df.rename(columns={
    "clean_text": "content",
    "title_clean": "title",
    "url":"link"
})
df['speaker'] = 'William Poole'

dfConsistent = df[["District","MultSpeakers","VideoForm","Article","content","title","speaker","location","link","date"]]
dfConsistent.head()

,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date
0,StLouis,0.0,False,False,Economic Growth: Is the Fed Irrelevant? St. Lo...,Economic Growth: Is the Fed Irrelevant?,William Poole,St. Louis Regional Commerce and Growth Associa...,https://fraser.stlouisfed.org/title/statements...,1998-07-15
1,StLouis,0.0,False,False,A Policymaker Confronts Uncertainty St. Louis ...,A Policymaker Confronts Uncertainty,William Poole,St. Louis Gateway Chapter of the National Asso...,https://fraser.stlouisfed.org/title/statements...,1998-09-16
2,StLouis,0.0,False,False,Is Inflation Too Low? 16th Annual Monetary Con...,Is Inflation Too Low?,William Poole,"16th Annual Monetary Conference, Cato Institut...",https://fraser.stlouisfed.org/title/statements...,1998-10-22
3,StLouis,0.0,False,False,Whither the U.S. Credit Markets? Construction ...,Whither the U.S. Credit Markets?,William Poole,"Construction Financial Managers Association, C...",https://fraser.stlouisfed.org/title/statements...,1998-10-26
4,StLouis,0.0,False,False,That Mysterious FOMC The Economic Club of Memp...,That Mysterious FOMC,William Poole,"The Economic Club of Memphis, Memphis, Tennessee",https://fraser.stlouisfed.org/title/statements...,1998-12-03


In [10]:
# Define the suspicious token checker
def is_suspicious_token(token, max_length):
    t = token.strip()

    # --- 1. Length filter (Fed speeches rarely have tokens > 40 chars) ---
    if len(t) > max_length:
        return True

    # --- 2. URL-like patterns ---
    url_patterns = [
        r'http', r'www', r'://', r'onepage',
        r'%\d{2}',            # URL-encoded characters like %20
        r'[?&=].+',           # query parameters
        r'/',                 # slash in a token = almost always URL/path
    ]

    # --- 3. File extensions (robust to punctuation or query strings) ---
    file_ext_pattern = r'\.(aspx|html?|php|pdf|txt|docx?|xls[xm]?|pptx?)\b'

    # --- 4. Slug-like tokens (4+ hyphens) ---
    slug_pattern = r'(?:[A-Za-z0-9]+-){4,}[A-Za-z0-9]+'

    # --- 5. Mixed-case PDF garbage ---
    # e.g., "ofBGoaorvdeorfnGorosveorfnothrseoFfethdeerFaeldRereasl"
    def looks_like_pdf_garbage(s):
        if len(s) < 25:
            return False
        uppers = sum(c.isupper() for c in s)
        lowers = sum(c.islower() for c in s)
        # High mixture of upper/lower in a long token = garbage
        return uppers > 5 and lowers > 5 and (uppers + lowers) > 0.8 * len(s)

    # --- 6. High symbol density (URL/query garbage) ---
    symbol_pattern = r'[&%#@+=]{2,}'  # 2+ symbols in a row

    # --- 7. Long alphanumeric junk ---
    alnum_junk_pattern = r'[A-Za-z]+\d+[A-Za-z]+'


    # --- Apply all patterns ---
    patterns = (
        url_patterns
        + [file_ext_pattern, slug_pattern, symbol_pattern, alnum_junk_pattern]
    )

    if any(re.search(p, t, flags=re.IGNORECASE) for p in patterns):
        return True

    if looks_like_pdf_garbage(t):
        return True

    return False

# Function to clean a single text entry
def clean_text(text, max_length=40):
    if not isinstance(text, str):
        return text  # Skip non-string entries
    tokens = text.split()
    filtered_tokens = [t for t in tokens if not is_suspicious_token(t, max_length)]
    return ' '.join(filtered_tokens)

# Example: Apply to a DataFrame column
# Assuming your DataFrame is called `df` and the column is `speech_text`
dfConsistent['content'] = dfConsistent['content'].apply(clean_text)


C:\Users\Lynn\AppData\Local\Temp\ipykernel_13240\4033248586.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfConsistent['content'] = dfConsistent['content'].apply(clean_text)


In [152]:
# # Checking for abnormality
# dfConsistent['tokens'] = dfConsistent['content'].apply(lambda x: str(x).split())

# threshold = 30
# dfConsistent['long_tokens'] = dfConsistent['tokens'].apply(lambda tokens: [t for t in tokens if len(t) > threshold])
# dfConsistent['has_long_token'] = dfConsistent['long_tokens'].apply(lambda x: len(x) > 0)

# suspicious_rows = dfConsistent[dfConsistent['has_long_token']]
# print(suspicious_rows.shape)

# suspicious_rows.head()


(1, 13)


C:\Users\Lynn\AppData\Local\Temp\ipykernel_18496\4154586137.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfConsistent['tokens'] = dfConsistent['content'].apply(lambda x: str(x).split())
C:\Users\Lynn\AppData\Local\Temp\ipykernel_18496\4154586137.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfConsistent['long_tokens'] = dfConsistent['tokens'].apply(lambda tokens: [t for t in tokens if len(t) > threshold])
C:\Users\Lynn\AppData\Local\Temp\ipykernel_18496\4154586137.py:6: SettingWithCopyWarning: 

,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,tokens,long_tokens,has_long_token
117,StLouis,0.0,False,False,Energy and the U.S. Macro Economy I am sure th...,Energy and the U.S. Macro Economy,William Poole,"Wilmington Club, Wilmington, Delaware",https://fraser.stlouisfed.org/title/statements...,2007-07-24,"[Energy, and, the, U.S., Macro, Economy, I, am...",[laeRforalloDrepnoitpmusnoCygrenE:8erugiF],True


In [154]:
# dfConsistent.drop(columns=['tokens','long_tokens','has_long_token'], errors = 'ignore', inplace=True)
# dfConsistent.columns.tolist()


C:\Users\Lynn\AppData\Local\Temp\ipykernel_18496\2290288276.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfConsistent.drop(columns=['tokens','long_tokens','has_long_token'], errors = 'ignore', inplace=True)


['District',
 'MultSpeakers',
 'VideoForm',
 'Article',
 'content',
 'title',
 'speaker',
 'location',
 'link',
 'date']

In [12]:
print(dfConsistent.shape)
dfConsistent.head()


(132, 10)


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date
0,StLouis,0.0,False,False,Economic Growth: Is the Fed Irrelevant? St. Lo...,Economic Growth: Is the Fed Irrelevant?,William Poole,St. Louis Regional Commerce and Growth Associa...,https://fraser.stlouisfed.org/title/statements...,1998-07-15
1,StLouis,0.0,False,False,A Policymaker Confronts Uncertainty St. Louis ...,A Policymaker Confronts Uncertainty,William Poole,St. Louis Gateway Chapter of the National Asso...,https://fraser.stlouisfed.org/title/statements...,1998-09-16
2,StLouis,0.0,False,False,Is Inflation Too Low? 16th Annual Monetary Con...,Is Inflation Too Low?,William Poole,"16th Annual Monetary Conference, Cato Institut...",https://fraser.stlouisfed.org/title/statements...,1998-10-22
3,StLouis,0.0,False,False,Whither the U.S. Credit Markets? Construction ...,Whither the U.S. Credit Markets?,William Poole,"Construction Financial Managers Association, C...",https://fraser.stlouisfed.org/title/statements...,1998-10-26
4,StLouis,0.0,False,False,That Mysterious FOMC The Economic Club of Memp...,That Mysterious FOMC,William Poole,"The Economic Club of Memphis, Memphis, Tennessee",https://fraser.stlouisfed.org/title/statements...,1998-12-03


## Run algorithms

In [20]:
# Now actually run all the algorithms, and just copy paste from now
modelFinance = pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis", 
                        top_k=None, truncation=True)

def positiveFin(doc):
    labels = ['positive','neutral','negative']
    # This is when there are sentences
    sentList = sent_tokenize(doc)
    
    # we loop since there are multiple paragraph for one doc, store them 
    result0 = [0 for i in range(len(labels))]
    for para in sentList:
        try:
            output = modelFinance(para)
            for i in range(len(labels)):
                d = output[0][i]
                for j, emo in enumerate(labels):
                    if d['label'] == emo: 
                        result0[j] += d['score']
        except Exception as e: 
            print(e)
            pass
    results = [i/len(sentList) for i in result0]
    return results

# apply to the statement:
dfConsistent['sentimentalFin'] = dfConsistent['content'].progress_apply(lambda x: positiveFin(x))

# Then expand it out:
labelList = ['positiveFin','neutralFin','negativeFin']
dfConsistent[labelList] = pd.DataFrame(dfConsistent.sentimentalFin.tolist(), index= dfConsistent.index)
dfConsistent.drop(columns=['sentimentalFin'], inplace=True)
dfConsistent.head()


Device set to use cpu
100%|██████████| 132/132 [11:06<00:00,  5.05s/it]


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,positiveFin,neutralFin,negativeFin
0,StLouis,0.0,False,False,Economic Growth: Is the Fed Irrelevant? St. Lo...,Economic Growth: Is the Fed Irrelevant?,William Poole,St. Louis Regional Commerce and Growth Associa...,https://fraser.stlouisfed.org/title/statements...,1998-07-15,0.214051,0.695975,0.089974
1,StLouis,0.0,False,False,A Policymaker Confronts Uncertainty St. Louis ...,A Policymaker Confronts Uncertainty,William Poole,St. Louis Gateway Chapter of the National Asso...,https://fraser.stlouisfed.org/title/statements...,1998-09-16,0.174800,0.716050,0.109150
2,StLouis,0.0,False,False,Is Inflation Too Low? 16th Annual Monetary Con...,Is Inflation Too Low?,William Poole,"16th Annual Monetary Conference, Cato Institut...",https://fraser.stlouisfed.org/title/statements...,1998-10-22,0.263236,0.548523,0.188241
3,StLouis,0.0,False,False,Whither the U.S. Credit Markets? Construction ...,Whither the U.S. Credit Markets?,William Poole,"Construction Financial Managers Association, C...",https://fraser.stlouisfed.org/title/statements...,1998-10-26,0.252848,0.546191,0.200960
4,StLouis,0.0,False,False,That Mysterious FOMC The Economic Club of Memp...,That Mysterious FOMC,William Poole,"The Economic Club of Memphis, Memphis, Tennessee",https://fraser.stlouisfed.org/title/statements...,1998-12-03,0.057943,0.931715,0.010342


In [21]:
import spacy

# Load the SpaCy model
nlp = spacy.load('en_core_web_sm')

# Define vocabularies
list_goal_voca = {"inflation", "growth", "price", "prices", "wages", "employment", "unemployment", "output", "activity", "GDP", "job"}
list_dovish_voca = {"cut", "decrease", "down", "ease", "fall", "reduce", "deflate", "weak", "weaken", "slow", "loss", "slowdown", "decelerate", "mitigate", "contract", "slice", "shave", "trim", "drop"}
list_hawkish_voca = {"increase", "higher", "raise", "rise", "tighten", "strong", "fast", "gain", "hike", "growth", "accelerate", "expand", "surge", "boom", "strengthen", "up", "lift", "climb"}
# list_negation_voca = {"not", "no", "never", "hardly", "barely"}. We use instead the empty set to avoid dealing with negation

list_negation_voca = {} 

# Define n_gram window
N_GRAM = 5

# Lemmatize statements using SpaCy to automatically tag POS before lemmatizations
def lemmatize_statement_spacy(statement):
    if statement is None or not isinstance(statement, str):  # Handle None values and floats
        return ''
    doc = nlp(statement)
    return ' '.join([token.lemma_ for token in doc])

# Compute the sentiment score with n-gram, handling negations and special treatment for 'unemployment'
def compute_FOMC_score(statement):
    statement = statement.replace(',', '').replace('.', '') # To exclude them from context windows
    words_list = statement.split()
    
    nb_dovish_w = 0
    nb_hawkish_w = 0

    for idx, word in enumerate(words_list):
        context_window = words_list[max(0, idx - N_GRAM // 2): idx + N_GRAM // 2 + 1]

        if word in list_goal_voca:
            negation_present = any(neg_word in context_window for neg_word in list_negation_voca)
            
            flag_dovish = False
            flag_hawkish = False

            for subword in context_window:
                if word != 'unemployment' and subword in list_dovish_voca:
                    if negation_present:
                        nb_hawkish_w += 1
                        flag_hawkish = True
                    else:
                        nb_dovish_w += 1
                        flag_dovish = True
                elif word != 'unemployment' and subword in list_hawkish_voca:
                    if negation_present:
                        nb_dovish_w += 1
                        flag_dovish = True
                    else:
                        nb_hawkish_w += 1
                        flag_hawkish = True
                elif word == 'unemployment' and subword in list_dovish_voca:
                    if negation_present:
                        nb_dovish_w += 1
                        flag_dovish = True
                    else: 
                        nb_hawkish_w += 1
                        flag_hawkish = True
                elif word == 'unemployment' and subword in list_hawkish_voca:
                    if negation_present:
                        nb_hawkish_w += 1
                        flag_hawkish = True
                    else: 
                        nb_dovish_w += 1 
                        flag_dovish = True
                else: 
                    pass

            if flag_dovish and flag_hawkish:
                nb_dovish_w -= 1
                nb_hawkish_w -= 1

    if nb_hawkish_w + nb_dovish_w != 0: 
        score = (nb_hawkish_w - nb_dovish_w) / (nb_hawkish_w + nb_dovish_w)
    else:
        score = 0
    return score, nb_hawkish_w, nb_dovish_w

# Adds hawkish scores to the dataset with n-gram choice using SpaCy lemmatization
def add_hawkish_score_spacy(df):
    scores = []
    scoreHawk = []
    scoreDove = []
    for statement in tq(df['content']):
        lemmatized_statement = lemmatize_statement_spacy(statement)
        score, hawk, dove = compute_FOMC_score(lemmatized_statement)
        scores.append(score)
        scoreHawk.append(hawk)
        scoreDove.append(dove)
    df['hawkishMal'] = scores
    df['n_hawk_pair'] = scoreHawk
    df['n_dove_pair'] = scoreDove
    return df

add_hawkish_score_spacy(dfConsistent)
dfConsistent.head()


100%|██████████| 132/132 [01:28<00:00,  1.49it/s]


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,positiveFin,neutralFin,negativeFin,hawkishMal,n_hawk_pair,n_dove_pair
0,StLouis,0.0,False,False,Economic Growth: Is the Fed Irrelevant? St. Lo...,Economic Growth: Is the Fed Irrelevant?,William Poole,St. Louis Regional Commerce and Growth Associa...,https://fraser.stlouisfed.org/title/statements...,1998-07-15,0.214051,0.695975,0.089974,1.00,23,0
1,StLouis,0.0,False,False,A Policymaker Confronts Uncertainty St. Louis ...,A Policymaker Confronts Uncertainty,William Poole,St. Louis Gateway Chapter of the National Asso...,https://fraser.stlouisfed.org/title/statements...,1998-09-16,0.174800,0.716050,0.109150,0.80,9,1
2,StLouis,0.0,False,False,Is Inflation Too Low? 16th Annual Monetary Con...,Is Inflation Too Low?,William Poole,"16th Annual Monetary Conference, Cato Institut...",https://fraser.stlouisfed.org/title/statements...,1998-10-22,0.263236,0.548523,0.188241,0.75,7,1
3,StLouis,0.0,False,False,Whither the U.S. Credit Markets? Construction ...,Whither the U.S. Credit Markets?,William Poole,"Construction Financial Managers Association, C...",https://fraser.stlouisfed.org/title/statements...,1998-10-26,0.252848,0.546191,0.200960,1.00,2,0
4,StLouis,0.0,False,False,That Mysterious FOMC The Economic Club of Memp...,That Mysterious FOMC,William Poole,"The Economic Club of Memphis, Memphis, Tennessee",https://fraser.stlouisfed.org/title/statements...,1998-12-03,0.057943,0.931715,0.010342,0.00,0,0


In [22]:
from wordtangible import word_concreteness, avg_text_concreteness, concrete_abstract_ratio

def concreteAvg(doc):
    result = avg_text_concreteness(doc)
    if result == 0.0:
        return np.nan
    else: 
        return result

def concreteAvgExtra(doc):
    result = avg_text_concreteness(doc, only_rated_words = False)
    if result == 0.0:
        return np.nan
    else: 
        return result

def concreteRatio(doc):
    result = concrete_abstract_ratio(doc)
    if result == np.inf:
        return 5.0
    else:
        return result

# apply to the statement:
dfConsistent['concreteness'] = dfConsistent['content'].progress_apply(lambda x: concreteAvg(x))
dfConsistent['concretenessExtra'] = dfConsistent['content'].progress_apply(lambda x: concreteAvgExtra(x))
dfConsistent['concreteRatio'] = dfConsistent['content'].progress_apply(lambda x: concreteRatio(x))

# Then expand it out:
dfConsistent.head()


100%|██████████| 132/132 [00:03<00:00, 36.87it/s]


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,positiveFin,neutralFin,negativeFin,hawkishMal,n_hawk_pair,n_dove_pair,concreteness,concretenessExtra,concreteRatio
0,StLouis,0.0,False,False,Economic Growth: Is the Fed Irrelevant? St. Lo...,Economic Growth: Is the Fed Irrelevant?,William Poole,St. Louis Regional Commerce and Growth Associa...,https://fraser.stlouisfed.org/title/statements...,1998-07-15,0.214051,0.695975,0.089974,1.00,23,0,2.816635,2.344225,0.556604
1,StLouis,0.0,False,False,A Policymaker Confronts Uncertainty St. Louis ...,A Policymaker Confronts Uncertainty,William Poole,St. Louis Gateway Chapter of the National Asso...,https://fraser.stlouisfed.org/title/statements...,1998-09-16,0.174800,0.716050,0.109150,0.80,9,1,2.783779,2.264870,0.495327
2,StLouis,0.0,False,False,Is Inflation Too Low? 16th Annual Monetary Con...,Is Inflation Too Low?,William Poole,"16th Annual Monetary Conference, Cato Institut...",https://fraser.stlouisfed.org/title/statements...,1998-10-22,0.263236,0.548523,0.188241,0.75,7,1,2.711066,2.017680,0.323326
3,StLouis,0.0,False,False,Whither the U.S. Credit Markets? Construction ...,Whither the U.S. Credit Markets?,William Poole,"Construction Financial Managers Association, C...",https://fraser.stlouisfed.org/title/statements...,1998-10-26,0.252848,0.546191,0.200960,1.00,2,0,2.761648,2.166606,0.457792
4,StLouis,0.0,False,False,That Mysterious FOMC The Economic Club of Memp...,That Mysterious FOMC,William Poole,"The Economic Club of Memphis, Memphis, Tennessee",https://fraser.stlouisfed.org/title/statements...,1998-12-03,0.057943,0.931715,0.010342,0.00,0,0,2.926811,2.233720,1.016575


In [40]:
# now deal with the nan on the average concrete
dfConsistent['concreteMean'] = dfConsistent['concreteness']
dfConsistent['concrete25'] = dfConsistent['concreteness']
dfConsistent['concreteMean'].fillna(dfConsistent['concreteMean'].mean(), inplace = True)
dfConsistent['concrete25'].fillna(2.5, inplace = True)

dfConsistent['concreteMeanExtra'] = dfConsistent['concretenessExtra']
dfConsistent['concrete25Extra'] = dfConsistent['concretenessExtra']
dfConsistent['concreteMeanExtra'].fillna(dfConsistent['concreteMeanExtra'].mean(), inplace = True)
dfConsistent['concrete25Extra'].fillna(2.5, inplace = True)

dfConsistent.head()


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,...,smog,gunning_fog,automated_readability_index,coleman_liau_index,lix,rix,concreteMean,concrete25,concreteMeanExtra,concrete25Extra
0,StLouis,0.0,False,False,Economic Growth: Is the Fed Irrelevant? St. Lo...,Economic Growth: Is the Fed Irrelevant?,William Poole,St. Louis Regional Commerce and Growth Associa...,https://fraser.stlouisfed.org/title/statements...,1998-07-15,...,9.879675,10.134013,7.595854,7.962236,35.172259,3.091463,2.816635,2.816635,2.344225,2.344225
1,StLouis,0.0,False,False,A Policymaker Confronts Uncertainty St. Louis ...,A Policymaker Confronts Uncertainty,William Poole,St. Louis Gateway Chapter of the National Asso...,https://fraser.stlouisfed.org/title/statements...,1998-09-16,...,13.247686,14.832333,13.272392,11.297273,49.135254,6.032680,2.783779,2.783779,2.264870,2.264870
2,StLouis,0.0,False,False,Is Inflation Too Low? 16th Annual Monetary Con...,Is Inflation Too Low?,William Poole,"16th Annual Monetary Conference, Cato Institut...",https://fraser.stlouisfed.org/title/statements...,1998-10-22,...,14.483606,15.946324,13.230582,12.706202,51.315391,6.405405,2.711066,2.711066,2.017680,2.017680
3,StLouis,0.0,False,False,Whither the U.S. Credit Markets? Construction ...,Whither the U.S. Credit Markets?,William Poole,"Construction Financial Managers Association, C...",https://fraser.stlouisfed.org/title/statements...,1998-10-26,...,11.771149,13.012710,11.784002,10.449360,44.779613,5.012821,2.761648,2.761648,2.166606,2.166606
4,StLouis,0.0,False,False,That Mysterious FOMC The Economic Club of Memp...,That Mysterious FOMC,William Poole,"The Economic Club of Memphis, Memphis, Tennessee",https://fraser.stlouisfed.org/title/statements...,1998-12-03,...,12.038527,12.759607,10.753850,10.803023,45.912396,5.135484,2.926811,2.926811,2.233720,2.233720


In [23]:
try:
    import textdescriptives
except:
    !pip install "textdescriptives[tutorials]"
    import textdescriptives
    
# more on https://hlasse.github.io/TextDescriptives/dependencydistance.html
# GitHub link https://github.com/HLasse/TextDescriptives/tree/main
import textdescriptives as td
import spacy

metrics = td.extract_metrics(
    text=dfConsistent["content"],
    spacy_model="en_core_web_sm",
    metrics=["descriptive_stats","readability", "dependency_distance","coherence","information_theory","quality"],
)

metrics_df = dfConsistent.join(metrics.drop(columns=["text"]))

print(metrics_df.shape)
print(metrics_df.columns.tolist())

dfConsistent = metrics_df.copy()


(132, 71)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'entropy', 'perplexity', 'per_word_perplexity', 'first_order_coherence', 'second_order_coherence', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'contains_lorem ipsum

## Export

In [42]:
print(dfConsistent.shape)
print(dfConsistent.columns.tolist())


(132, 75)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'entropy', 'perplexity', 'per_word_perplexity', 'first_order_coherence', 'second_order_coherence', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'contains_lorem ipsum

In [44]:
dfConsistent.to_csv('WilliamPooleNewVarExtended.csv', date_format='%Y-%m-%d', index = False)


# I need to remove links, unreasonably long tokens

In [254]:
# import the whole thing, now remove everything except the 10 original variables.
df = pd.read_csv('FedSpeechesTotalExtended.csv', parse_dates = ['date'])
print(df.shape)
df.head()


(7108, 10)


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16


In [256]:
# Define the suspicious token checker
def is_suspicious_token(token, max_length):
    t = token.strip()

    # --- 1. Length filter (Fed speeches rarely have tokens > 40 chars) ---
    if len(t) > max_length:
        return True

    # --- 2. URL-like patterns ---
    url_patterns = [
        r'http', r'www', r'://', r'onepage',
        r'%\d{2}',            # URL-encoded characters like %20
        r'[?&=].+',           # query parameters
        r'/',                 # slash in a token = almost always URL/path
    ]

    # --- 3. File extensions (robust to punctuation or query strings) ---
    file_ext_pattern = r'\.(aspx|html?|php|pdf|txt|docx?|xls[xm]?|pptx?)\b'

    # --- 4. Slug-like tokens (4+ hyphens) ---
    slug_pattern = r'(?:[A-Za-z0-9]+-){4,}[A-Za-z0-9]+'

    # --- 5. Mixed-case PDF garbage ---
    # e.g., "ofBGoaorvdeorfnGorosveorfnothrseoFfethdeerFaeldRereasl"
    def looks_like_pdf_garbage(s):
        if len(s) < 25:
            return False
        uppers = sum(c.isupper() for c in s)
        lowers = sum(c.islower() for c in s)
        # High mixture of upper/lower in a long token = garbage
        return uppers > 5 and lowers > 5 and (uppers + lowers) > 0.8 * len(s)

    # --- 6. High symbol density (URL/query garbage) ---
    symbol_pattern = r'[&%#@+=]{2,}'  # 2+ symbols in a row

    # --- 7. Long alphanumeric junk ---
    alnum_junk_pattern = r'[A-Za-z]+\d+[A-Za-z]+'


    # --- Apply all patterns ---
    patterns = (
        url_patterns
        + [file_ext_pattern, slug_pattern, symbol_pattern, alnum_junk_pattern]
    )

    if any(re.search(p, t, flags=re.IGNORECASE) for p in patterns):
        return True

    if looks_like_pdf_garbage(t):
        return True

    return False

# Function to clean a single text entry
def clean_text(text, max_length=40):
    if not isinstance(text, str):
        return text  # Skip non-string entries
    tokens = text.split()
    filtered_tokens = [t for t in tokens if not is_suspicious_token(t, max_length)]
    return ' '.join(filtered_tokens)

# Example: Apply to a DataFrame column
# Assuming your DataFrame is called `df` and the column is `speech_text`
df['content'] = df['content'].apply(clean_text)

In [258]:
df.to_csv('FedSpeechesTotalExtendedCleaned.csv', date_format='%Y-%m-%d', index = False)


# Extracting textual variables

In [16]:
from transformers import pipeline
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from nltk.tokenize import word_tokenize, sent_tokenize


In [17]:
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm as tq
tq.pandas() #thanks to https://stackoverflow.com/questions/18603270/progress-indicator-during-pandas-operations

## Importing data

In [288]:
df = pd.read_csv('FedSpeechesTotalExtendedCleaned.csv', parse_dates = ['date'])
print(df.shape)
df.head()


(7108, 10)


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16


In [289]:
# check for long tokens
# checking if there are abnormal token length that got drowned out by the long documents
df['tokens'] = df['content'].apply(lambda x: str(x).split())

threshold = 30
df['long_tokens'] = df['tokens'].apply(lambda tokens: [t for t in tokens if len(t) > threshold])
df['has_long_token'] = df['long_tokens'].apply(lambda x: len(x) > 0)

suspicious_rows = df[df['has_long_token']]
print(suspicious_rows.shape)

suspicious_rows.head()
# df['num_long_tokens'] = df['long_tokens'].apply(len)
# plt.hist(df['num_long_tokens'], bins=30)
# plt.title("Long Token Count per Row")
# plt.xlabel("Number of Long Tokens")
# plt.ylabel("Frequency")
# plt.show()



(312, 13)


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,tokens,long_tokens,has_long_token
90,Philadelphia,0.0,False,False,"Good afternoon and, once again, welcome to the...",A Look at Fintech from the Inside to the Upside,Patrick T. Harker,Fintech and Financial Institutions Conference ...,https://www.philadelphiafed.org/the-economy/ba...,2025-04-10,"[Good, afternoon, and,, once, again,, welcome,...",[capital-productivity-augmenting],True
95,Philadelphia,0.0,False,False,"Good evening, everyone, and thank you for bein...","Fintech, AI, and the Changing Financial Landscape",Patrick T. Harker,Carnegie Mellon University Lecture Series | Pi...,https://www.philadelphiafed.org/the-economy/ba...,2024-11-12,"[Good, evening,, everyone,, and, thank, you, f...",[capital-productivity-augmenting],True
288,BoardGovernors,0.0,False,False,"Thank you, Athanasios, and thank you for the o...",Thoughts on the Economy and Policy Rules at th...,Governor Christopher J. Waller,At “A 50 Year Retrospective on the Shadow Open...,https://www.federalreserve.gov/newsevents/spee...,2024-10-14,"[Thank, you,, Athanasios,, and, thank, you, fo...","[Persistence,""Carnegie-Rochester]",True
305,BoardGovernors,0.0,False,False,Thank you for the invitation to join you.1Give...,The Future of Stress Testing and the Stress Ca...,Governor Michelle W. Bowman,At the Executive Council of the Banking Law Se...,https://www.federalreserve.gov/newsevents/spee...,2024-09-10,"[Thank, you, for, the, invitation, to, join, y...",[approachrequireacross-the-board],True
312,BoardGovernors,0.0,False,False,Thank you for the invitation to join you again...,"Update on the Economic Outlook, and Perspectiv...",Governor Michelle W. Bowman,At the 2024 CEO and Senior Management Summit a...,https://www.federalreserve.gov/newsevents/spee...,2024-08-10,"[Thank, you, for, the, invitation, to, join, y...","[change,supervisorycommunications]",True


In [291]:
suspicious_rows[['long_tokens']].head(20)

,long_tokens
90,[capital-productivity-augmenting]
95,[capital-productivity-augmenting]
288,"[Persistence,""Carnegie-Rochester]"
305,[approachrequireacross-the-board]
312,"[change,supervisorycommunications]"
341,[Developingconsensus-basedstandards]
351,"[Coefficients,""Econometrica,vol.]"
363,[sooner-than-previously-anticipated]
438,"[Companies,""FederalRegister,vol.]"
448,"[Interpretation,""Econometrica,vol., Fluctuatio..."


In [294]:
df.drop(columns=['tokens','long_tokens','has_long_token'], errors = 'ignore', inplace=True)
df.columns.tolist()

['District',
 'MultSpeakers',
 'VideoForm',
 'Article',
 'content',
 'title',
 'speaker',
 'location',
 'link',
 'date']

## Positive/Negative Financial

In [296]:
# https://huggingface.co/mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis
modelFinance = pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis", 
                        top_k=None, truncation=True)
# modelFinanceTrunc = pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis", 
#                         top_k=None, truncation=True)


Device set to use cpu


In [298]:
def positiveFin(doc):
    labels = ['positive','neutral','negative']
    # This is when there are sentences
    sentList = sent_tokenize(doc)
    
    # we loop since there are multiple paragraph for one doc, store them 
    result0 = [0 for i in range(len(labels))]
    for para in sentList:
        try:
            output = modelFinance(para)
            for i in range(len(labels)):
                d = output[0][i]
                for j, emo in enumerate(labels):
                    if d['label'] == emo: 
                        result0[j] += d['score']
        except Exception as e: 
            print(e)
            pass
    results = [i/len(sentList) for i in result0]
    return results
    

In [300]:
# apply to the statement:
df['sentimentalFin'] = df['content'].progress_apply(lambda x: positiveFin(x))

# Then expand it out:
labelList = ['positiveFin','neutralFin','negativeFin']
df[labelList] = pd.DataFrame(df.sentimentalFin.tolist(), index= df.index)
df.drop(columns=['sentimentalFin'], inplace=True)
df.head()


100%|██████████| 7108/7108 [7:09:34<00:00,  3.63s/it]   


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,positiveFin,neutralFin,negativeFin
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,0.408672,0.320187,0.271142
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,0.083191,0.874509,0.042300
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,0.327147,0.590322,0.082531
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,0.467524,0.391142,0.141334
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,0.228169,0.700974,0.070857


## Hawkish

In [302]:
import spacy

# Load the SpaCy model
nlp = spacy.load('en_core_web_sm')

# Define vocabularies
list_goal_voca = {"inflation", "growth", "price", "prices", "wages", "employment", "unemployment", "output", "activity", "GDP", "job"}
list_dovish_voca = {"cut", "decrease", "down", "ease", "fall", "reduce", "deflate", "weak", "weaken", "slow", "loss", "slowdown", "decelerate", "mitigate", "contract", "slice", "shave", "trim", "drop"}
list_hawkish_voca = {"increase", "higher", "raise", "rise", "tighten", "strong", "fast", "gain", "hike", "growth", "accelerate", "expand", "surge", "boom", "strengthen", "up", "lift", "climb"}
# list_negation_voca = {"not", "no", "never", "hardly", "barely"}. We use instead the empty set to avoid dealing with negation

list_negation_voca = {} 

# Define n_gram window
N_GRAM = 5

# Lemmatize statements using SpaCy to automatically tag POS before lemmatizations
def lemmatize_statement_spacy(statement):
    if statement is None or not isinstance(statement, str):  # Handle None values and floats
        return ''
    doc = nlp(statement)
    return ' '.join([token.lemma_ for token in doc])

# Compute the sentiment score with n-gram, handling negations and special treatment for 'unemployment'
def compute_FOMC_score(statement):
    statement = statement.replace(',', '').replace('.', '') # To exclude them from context windows
    words_list = statement.split()
    
    nb_dovish_w = 0
    nb_hawkish_w = 0

    for idx, word in enumerate(words_list):
        context_window = words_list[max(0, idx - N_GRAM // 2): idx + N_GRAM // 2 + 1]

        if word in list_goal_voca:
            negation_present = any(neg_word in context_window for neg_word in list_negation_voca)
            
            flag_dovish = False
            flag_hawkish = False

            for subword in context_window:
                if word != 'unemployment' and subword in list_dovish_voca:
                    if negation_present:
                        nb_hawkish_w += 1
                        flag_hawkish = True
                    else:
                        nb_dovish_w += 1
                        flag_dovish = True
                elif word != 'unemployment' and subword in list_hawkish_voca:
                    if negation_present:
                        nb_dovish_w += 1
                        flag_dovish = True
                    else:
                        nb_hawkish_w += 1
                        flag_hawkish = True
                elif word == 'unemployment' and subword in list_dovish_voca:
                    if negation_present:
                        nb_dovish_w += 1
                        flag_dovish = True
                    else: 
                        nb_hawkish_w += 1
                        flag_hawkish = True
                elif word == 'unemployment' and subword in list_hawkish_voca:
                    if negation_present:
                        nb_hawkish_w += 1
                        flag_hawkish = True
                    else: 
                        nb_dovish_w += 1 
                        flag_dovish = True
                else: 
                    pass

            if flag_dovish and flag_hawkish:
                nb_dovish_w -= 1
                nb_hawkish_w -= 1

    if nb_hawkish_w + nb_dovish_w != 0: 
        score = (nb_hawkish_w - nb_dovish_w) / (nb_hawkish_w + nb_dovish_w)
    else:
        score = 0
    return score, nb_hawkish_w, nb_dovish_w

# Adds hawkish scores to the dataset with n-gram choice using SpaCy lemmatization
def add_hawkish_score_spacy(df):
    scores = []
    scoreHawk = []
    scoreDove = []
    for statement in tq(df['content']):
        lemmatized_statement = lemmatize_statement_spacy(statement)
        score, hawk, dove = compute_FOMC_score(lemmatized_statement)
        scores.append(score)
        scoreHawk.append(hawk)
        scoreDove.append(dove)
    df['hawkishMal'] = scores
    df['n_hawk_pair'] = scoreHawk
    df['n_dove_pair'] = scoreDove
    return df


In [303]:
# add_hawkish_score_spacy(df)
add_hawkish_score_spacy(df)
df.head()


100%|██████████| 7108/7108 [58:31<00:00,  2.02it/s]  


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,positiveFin,neutralFin,negativeFin,hawkishMal,n_hawk_pair,n_dove_pair
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,0.408672,0.320187,0.271142,0.818182,10,1
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,0.083191,0.874509,0.042300,0.000000,0,0
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,0.327147,0.590322,0.082531,0.333333,4,2
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,0.467524,0.391142,0.141334,0.705882,29,5
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,0.228169,0.700974,0.070857,0.695652,39,7


## CentralbankRoBERTa

In [11]:
modelAgent = pipeline("text-classification", 
                      model="Moritz-Pfeifer/CentralBankRoBERTa-agent-classifier",
                      top_k=None, truncation=True)
# modelAgentTrunc = pipeline("text-classification", 
#                       model="Moritz-Pfeifer/CentralBankRoBERTa-agent-classifier",
#                       top_k=None, truncation=True)

modelSentiment = pipeline("text-classification", 
                      model="Moritz-Pfeifer/CentralBankRoBERTa-sentiment-classifier",
                      top_k = None, truncation=True)
# modelSentimentTrunc = pipeline("text-classification", 
#                       model="Moritz-Pfeifer/CentralBankRoBERTa-sentiment-classifier",
#                       top_k = None, truncation=True)

def truncate_sentence_nltk(sentence, max_tokens):
    # Tokenize the sentence using NLTK
    tokens = word_tokenize(sentence)
    # Truncate the tokens
    truncated_tokens = tokens[:max_tokens]
    # Join the tokens back into a sentence
    truncated_sentence = ' '.join(truncated_tokens)
    return truncated_sentence

# Central bank targets
def AgentClassifier(doc, n = 300):
    labels = ['Financial Sector','Central Bank','Government','Households','Firms']
    
    # check if the length (number of potential tokens) of the text exceed n:
    if len(word_tokenize(doc)) > n:
        # print("This text is too long")
        
        # This is when there are sentences
        # if it is, we divide it into pieces that is a little larger than 500 (can change this number if the error still there)
        sentList = sent_tokenize(doc)
        sentLen = 0
        sentPara = [""]
        for sent in sentList:
            if sentLen + len(word_tokenize(sent)) < n:
                sentLen += len(word_tokenize(sent))
                sentPara[-1] = " ".join([sentPara[-1], sent])
            else:
                sentLen = len(word_tokenize(sent))
                sentPara.append(sent)
        
        # we loop since there are multiple paragraph for one doc, store them 
        result0 = [0 for i in range(len(labels))]
        for para in sentPara:
            try:
                output = modelAgent(para)
                for i in range(len(labels)):
                    d = output[0][i]
                    for j, emo in enumerate(labels):
                        if d['label'] == emo: 
                            result0[j] += d['score']
            except Exception as e: 
                print(e)
                result0 = [0,0,0,0,0]
        results = [i/len(sentPara) for i in result0]
    else:
        # the doc itself is only a text
        try:
            output = modelAgent(doc)
            results = [0 for i in range(len(labels))]
            for i in range(len(labels)):
                d = output[0][i]
                for j, emo in enumerate(labels):
                    if d['label'] == emo: 
                        results[j] += d['score']
        except Exception as e: 
            print(e)
            result0 = [[0,0,0,0,0]]
    return results

def SentimentClassifier(doc,n =300):
    labels = ['positive','negative']
    # check if the length (number of potential tokens) of the text exceed n:
    if len(word_tokenize(doc)) > n:
        # print("This text is too long")
        
        # This is when there are sentences
        # if it is, we divide it into pieces that is a little larger than 500 (can change this number if the error still there)
        sentList = sent_tokenize(doc)
        sentLen = 0
        sentPara = [""]
        for sent in sentList:
            if sentLen + len(word_tokenize(sent)) < n:
                sentLen += len(word_tokenize(sent))
                sentPara[-1] = " ".join([sentPara[-1], sent])
            else:
                sentLen = len(word_tokenize(sent))
                sentPara.append(sent)
        
        # we loop since there are multiple paragraph for one doc, store them 
        result0 = [0 for i in range(len(labels))]
        for para in sentPara:
            try:
                output = modelSentiment(para)
                for i in range(len(labels)):
                    d = output[0][i]
                    for j, emo in enumerate(labels):
                        if d['label'] == emo: 
                            result0[j] += d['score']
            except Exception as e: 
                print(e)
                result0 = [0,0]
        results = [i/len(sentPara) for i in result0]
    else:
        # the doc itself is only a text
        try:
            output = modelSentiment(doc)
            results = [0 for i in range(len(labels))]
            for i in range(len(labels)):
                d = output[0][i]
                for j, emo in enumerate(labels):
                    if d['label'] == emo: 
                        results[j] += d['score']
        except Exception as e: 
            print(e)
            result0 = [0,0]
        
    return results


Device set to use cpu
Device set to use cpu


In [13]:
modelAgent = pipeline("text-classification", 
                      model="Moritz-Pfeifer/CentralBankRoBERTa-agent-classifier",
                      top_k=None, max_length = 514, truncation=True)
modelSentiment = pipeline("text-classification", 
                      model="Moritz-Pfeifer/CentralBankRoBERTa-sentiment-classifier",
                      top_k = None, max_length = 514,  truncation=True)

# Central bank targets
def AgentClassifier(doc):
    labels = ['Financial Sector','Central Bank','Government','Households','Firms']    
    sentList = sent_tokenize(doc)
    
    result0 = [0 for i in range(len(labels))]
    for para in sentList:
        try:
            output = modelAgent(para)
            for i in range(len(labels)):
                d = output[0][i]
                for j, emo in enumerate(labels):
                    if d['label'] == emo: 
                        result0[j] += d['score']
        except Exception as e: 
            print(e)
            pass
    results = [i/len(sentList) for i in result0]
    return results

def SentimentClassifier(doc,n =300):
    labels = ['positive','negative']
    sentList = sent_tokenize(doc)

    result0 = [0 for i in range(len(labels))]
    for para in sentList:
        try:
            output = modelSentiment(para)
            for i in range(len(labels)):
                d = output[0][i]
                for j, emo in enumerate(labels):
                    if d['label'] == emo: 
                        result0[j] += d['score']
        except Exception as e: 
            print(e)
            pass
    results = [i/len(sentList) for i in result0]        
    return results


Device set to use cpu
Device set to use cpu


In [15]:
# apply to the statement:
df['AgentScore'] = df['content'].progress_apply(lambda x: AgentClassifier(x))

# Then expand it out:
agentList = ['Financial Sector','Central Bank','Government','Households','Firms']
df[agentList] = pd.DataFrame(df.AgentScore.tolist(), index= df.index)
df.drop(columns=['AgentScore'], inplace=True)
df.head()


 18%|█▊        | 1238/6993 [1:58:15<7:40:44,  4.80s/it] 

The expanded size of the tensor (613) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 613].  Tensor sizes: [1, 514]


 22%|██▏       | 1524/6993 [2:27:01<8:47:37,  5.79s/it] 


KeyboardInterrupt: 

In [205]:
# apply to the statement:
df['SentimentScore'] = df['content'].progress_apply(lambda x: SentimentClassifier(x))

# Then expand it out:
sentimentList = ['positiveT','negativeT']
df[sentimentList] = pd.DataFrame(df.SentimentScore.tolist(), index= df.index)
df.drop(columns=['SentimentScore'], inplace=True)
df.head()


100%|██████████| 385/385 [37:49<00:00,  5.89s/it]  


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,...,hawkishMal,n_hawk_pair,n_dove_pair,Financial Sector,Central Bank,Government,Households,Firms,positiveT,negativeT
0,Boston,False,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,...,0.818182,10,1,0.114230,0.208634,0.099826,0.258446,0.318863,0.394824,0.605176
1,Boston,False,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,...,0.000000,0,0,0.115523,0.280729,0.164466,0.330763,0.108519,0.640235,0.359765
2,Boston,False,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,...,0.333333,4,2,0.099487,0.289442,0.070276,0.385861,0.154933,0.666246,0.333754
3,Boston,False,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,...,0.705882,29,5,0.077206,0.294306,0.035203,0.345561,0.247724,0.448353,0.551647
4,Boston,True,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,...,0.695652,39,7,0.191520,0.292865,0.057089,0.247316,0.211210,0.497653,0.502347


## Concreteness

In [306]:
from wordtangible import word_concreteness, avg_text_concreteness, concrete_abstract_ratio


In [308]:
def concreteAvg(doc):
    result = avg_text_concreteness(doc)
    if result == 0.0:
        return np.nan
    else: 
        return result

def concreteAvgExtra(doc):
    result = avg_text_concreteness(doc, only_rated_words = False)
    if result == 0.0:
        return np.nan
    else: 
        return result

def concreteRatio(doc):
    result = concrete_abstract_ratio(doc)
    if result == np.inf:
        return 5.0
    else:
        return result


In [310]:
# apply to the statement:
df['concreteness'] = df['content'].progress_apply(lambda x: concreteAvg(x))
df['concretenessExtra'] = df['content'].progress_apply(lambda x: concreteAvgExtra(x))
df['concreteRatio'] = df['content'].progress_apply(lambda x: concreteRatio(x))

# Then expand it out:
df.head()


100%|██████████| 7108/7108 [02:12<00:00, 53.58it/s]


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,positiveFin,neutralFin,negativeFin,hawkishMal,n_hawk_pair,n_dove_pair,concreteness,concretenessExtra,concreteRatio
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,0.408672,0.320187,0.271142,0.818182,10,1,2.784879,2.162424,0.427184
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,0.083191,0.874509,0.042300,0.000000,0,0,2.799398,2.103503,0.593750
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,0.327147,0.590322,0.082531,0.333333,4,2,2.826248,2.135829,0.617021
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,0.467524,0.391142,0.141334,0.705882,29,5,2.815404,2.197452,0.514599
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,0.228169,0.700974,0.070857,0.695652,39,7,2.764086,2.104119,0.401304


In [311]:
# now deal with the nan on the average concrete
df['concreteMean'] = df['concreteness']
df['concrete25'] = df['concreteness']
df['concreteMean'].fillna(df['concreteMean'].mean(), inplace = True)
df['concrete25'].fillna(2.5, inplace = True)

df['concreteMeanExtra'] = df['concretenessExtra']
df['concrete25Extra'] = df['concretenessExtra']
df['concreteMeanExtra'].fillna(df['concreteMeanExtra'].mean(), inplace = True)
df['concrete25Extra'].fillna(2.5, inplace = True)

df.head(10)


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,...,hawkishMal,n_hawk_pair,n_dove_pair,concreteness,concretenessExtra,concreteRatio,concreteMean,concrete25,concreteMeanExtra,concrete25Extra
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,...,0.818182,10,1,2.784879,2.162424,0.427184,2.784879,2.784879,2.162424,2.162424
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,...,0.000000,0,0,2.799398,2.103503,0.593750,2.799398,2.799398,2.103503,2.103503
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,...,0.333333,4,2,2.826248,2.135829,0.617021,2.826248,2.826248,2.135829,2.135829
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,...,0.705882,29,5,2.815404,2.197452,0.514599,2.815404,2.815404,2.197452,2.197452
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,...,0.695652,39,7,2.764086,2.104119,0.401304,2.764086,2.764086,2.104119,2.104119
5,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,"The Importance of a Patient, Methodical, and H...",Susan M. Collins,"New York, New York",https://www.bostonfed.org/news-and-events/spee...,2024-04-11,...,0.809524,19,2,2.750342,2.171486,0.364662,2.750342,2.750342,2.171486,2.171486
6,Boston,0.0,False,False,Good morning. It is a great pleasure to welcom...,Welcoming Remarks at the “Conference on the Fi...,Susan M. Collins,Virtual,https://www.bostonfed.org/news-and-events/spee...,2024-04-05,...,1.000000,1,0,2.944919,1.804007,0.934426,2.944919,2.944919,1.804007,1.804007
7,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,Observations on the Economy’s Performance and ...,Susan M. Collins,Tuck School of Business at Dartmouth College,https://www.bostonfed.org/news-and-events/spee...,2024-02-28,...,0.800000,36,4,2.828795,2.237142,0.608511,2.828795,2.828795,2.237142,2.237142
8,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,"The Economy’s Performance and Outlook, and Imp...",Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-02-07,...,0.636364,9,2,2.783275,2.167711,0.493506,2.783275,2.783275,2.167711,2.167711
9,Boston,0.0,False,False,It is a pleasure to take part in this conferen...,Opening Remarks for Sessions on Men and Women ...,Susan M. Collins,Virtual,https://www.bostonfed.org/news-and-events/spee...,2024-02-06,...,0.000000,1,1,2.920975,2.220451,0.728395,2.920975,2.920975,2.220451,2.220451


## Other complexity measurement

In [312]:
try:
    import textdescriptives
except:
    !pip install "textdescriptives[tutorials]"
    
# more on https://hlasse.github.io/TextDescriptives/dependencydistance.html
# GitHub link https://github.com/HLasse/TextDescriptives/tree/main
import textdescriptives as td
import spacy

In [321]:
# text quality
# assumes you have downloaded the ´en_core_web_model´
metrics = td.extract_metrics(
    text=df["content"],
    spacy_model="en_core_web_sm",
    metrics=["descriptive_stats","readability", "dependency_distance","coherence","information_theory","quality"],
)
# alternatively, you can specify the language and (optionally) the model size
# metrics = td.extract_metrics(text=df["message"], lang="en", spacy_model_size="sm", metrics=["readability, dependency_distance"])

# join the metrics to the original dataframe to get the label
metrics_df = df.join(metrics.drop(columns=["text"]))
metrics_df.head()


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,...,flesch_reading_ease,flesch_kincaid_grade,smog,gunning_fog,automated_readability_index,coleman_liau_index,lix,rix,first_order_coherence,second_order_coherence
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,...,44.329367,13.980255,15.169125,17.509585,17.256897,14.084854,60.426681,9.069767,0.526444,0.511636
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,...,46.478313,12.516994,14.296648,15.858004,14.896570,13.845811,56.337437,7.678571,0.387757,0.416154
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,...,54.190587,10.954297,13.161301,14.292075,13.416125,13.102487,52.760610,6.685315,0.381271,0.397782
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,...,50.613530,11.944837,14.295085,15.858275,14.835373,13.758830,53.964692,7.131737,0.452981,0.446677
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,...,71.294623,9.308595,11.003577,12.800382,10.815551,8.169472,42.031853,4.320000,0.375190,0.361304


In [322]:
print(metrics_df.shape)
print(metrics_df.columns.tolist())


(7108, 75)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'entropy', 'perplexity', 'per_word_perplexity', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', '

In [323]:
df = metrics_df.copy()

## Exporting data

In [327]:
print(df.shape)
print(df.columns.tolist())

(7108, 75)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'entropy', 'perplexity', 'per_word_perplexity', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', '

In [329]:
df.to_csv('FedSpeechesWithNewVarExtended.csv', date_format='%Y-%m-%d', index = False)


## Exporting data on original data (if needed)

In [20]:
# # df = pd.read_csv('FedSpeechesWithNewVar.csv', parse_dates = ['date'])
# dfOg = pd.read_csv('FedSpeechesWithNewVarExtended.csv', parse_dates = ['date'])
# print(dfOg.shape)
# print(dfOg.columns.tolist())

(6993, 72)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'first_order_coherence', 'second_order_coherence', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'contains_lorem ipsum', 'duplicate_line_chr_fraction', 'duplicate_paragraph_chr_fraction', 'duplicate_ngram_chr_fraction_5', 'duplicate_ngram_chr_fraction_6', 'duplicate_ngram_chr_fraction_7', 'duplicate_ngram_chr_fraction_8', 'duplicate_ngram_chr_fraction_9', 'duplicate_ngram_chr_fraction_10', 'top_ngram_chr_fraction_2', 'top_ngram_chr_fraction_3', 't

In [28]:
# df2 = df[['positiveFin', 'neutralFin', 'negativeFin']]
# merged_df = pd.concat([dfOg, df2], axis=1)
# print(merged_df.shape)
# print(merged_df.columns.tolist())


(6993, 75)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'first_order_coherence', 'second_order_coherence', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'contains_lorem ipsum', 'duplicate_line_chr_fraction', 'duplicate_paragraph_chr_fraction', 'duplicate_ngram_chr_fraction_5', 'duplicate_ngram_chr_fraction_6', 'duplicate_ngram_chr_fraction_7', 'duplicate_ngram_chr_fraction_8', 'duplicate_ngram_chr_fraction_9', 'duplicate_ngram_chr_fraction_10', 'top_ngram_chr_fraction_2', 'top_ngram_chr_fraction_3', 't

In [30]:
merged_df.head()

,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,...,flesch_kincaid_grade,smog,gunning_fog,automated_readability_index,coleman_liau_index,lix,rix,positiveFin,neutralFin,negativeFin
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,...,14.284428,15.311616,17.829265,17.549785,13.880232,61.046673,9.285714,0.397795,0.320107,0.282097
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,...,12.527884,14.296648,15.874501,14.851959,13.708817,56.276114,7.678571,0.083191,0.874509,0.042300
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,...,10.916871,13.126406,14.262921,13.263271,12.861831,52.498374,6.638889,0.326730,0.590644,0.082626
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,...,11.864655,14.230762,15.767775,14.627049,13.570251,53.562660,7.029412,0.466866,0.397021,0.136113
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,...,9.309457,11.003577,12.801361,10.816812,8.168961,42.032791,4.320000,0.227573,0.701712,0.070715


In [32]:
merged_df.to_csv('FedSpeechesWithNewVarExtended.csv', date_format='%Y-%m-%d', index = False)


## Add William Poole

In [46]:
dfOg = pd.read_csv('FedSpeechesWithNewVarExtended.csv', parse_dates = ['date'])
print(dfOg.shape)
print(dfOg.columns.tolist())


(7108, 75)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'entropy', 'perplexity', 'per_word_perplexity', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', '

In [50]:
# new data
dfWP = pd.read_csv('WilliamPooleNewVarExtended.csv', parse_dates = ['date'])
print(dfWP.shape)
dfWP = dfWP[dfOg.columns]
print(dfWP.columns.tolist())


(132, 75)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'entropy', 'perplexity', 'per_word_perplexity', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'c

In [54]:
# if they are the exact same:
df_all = pd.concat([dfOg, dfWP], ignore_index=True)
print(df_all.shape)
print(df_all.columns.tolist())


(7240, 75)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'content', 'title', 'speaker', 'location', 'link', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'entropy', 'perplexity', 'per_word_perplexity', 'dependency_distance_mean', 'dependency_distance_std', 'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', '

In [56]:
df_all.head()


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,...,flesch_reading_ease,flesch_kincaid_grade,smog,gunning_fog,automated_readability_index,coleman_liau_index,lix,rix,first_order_coherence,second_order_coherence
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,...,44.329367,13.980255,15.169125,17.509585,17.256897,14.084854,60.426681,9.069767,0.526444,0.511636
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,...,46.478313,12.516994,14.296648,15.858004,14.896570,13.845811,56.337437,7.678571,0.387757,0.416154
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,...,54.190587,10.954297,13.161301,14.292075,13.416125,13.102487,52.760610,6.685315,0.381271,0.397782
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,...,50.613530,11.944837,14.295085,15.858275,14.835373,13.758830,53.964692,7.131737,0.452981,0.446677
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,...,71.294623,9.308595,11.003577,12.800382,10.815551,8.169472,42.031853,4.320000,0.375190,0.361304


In [58]:
df_all.to_csv('FedSpeechesWithNewVarExtended.csv', date_format='%Y-%m-%d', index = False)


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt


In [2]:
# df = pd.read_csv('FedSpeechesWithNewVar.csv', parse_dates = ['date'])
df = pd.read_csv('FedSpeechesWithNewVarExtended.csv', parse_dates = ['date'])
print(df.shape)
df.head()

(7108, 75)


,District,MultSpeakers,VideoForm,Article,content,title,speaker,location,link,date,...,flesch_reading_ease,flesch_kincaid_grade,smog,gunning_fog,automated_readability_index,coleman_liau_index,lix,rix,first_order_coherence,second_order_coherence
0,Boston,0.0,False,False,"Economic Resilience, Amid Elevated Tariffs and...",Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",https://www.bostonfed.org/news-and-events/spee...,2025-06-25,...,44.329367,13.980255,15.169125,17.509585,17.256897,14.084854,60.426681,9.069767,0.526444,0.511636
1,Boston,0.0,False,False,I am pleased to welcome all of you to the Fede...,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-09-30,...,46.478313,12.516994,14.296648,15.858004,14.896570,13.845811,56.337437,7.678571,0.387757,0.416154
2,Boston,0.0,False,False,Takeaways from Boston Fed President Susan M. C...,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-06-18,...,54.190587,10.954297,13.161301,14.292075,13.416125,13.102487,52.760610,6.685315,0.381271,0.397782
3,Boston,0.0,False,False,Boston Fed President and CEO Susan M. Collins'...,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",https://www.bostonfed.org/news-and-events/spee...,2024-05-08,...,50.613530,11.944837,14.295085,15.858275,14.835373,13.758830,53.964692,7.131737,0.452981,0.446677
4,Boston,1.0,True,False,"hi, good afternoon everyone. my name is Beth B...",Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,https://www.bostonfed.org/news-and-events/spee...,2024-04-16,...,71.294623,9.308595,11.003577,12.800382,10.815551,8.169472,42.031853,4.320000,0.375190,0.361304
